# Priority Score: company data pipeline (v0.4)

**Part 1 of the Marketing Intelligence pipeline.** Merges and formats **HubSpot** (prospects) and **MyStaff** (active clients)
into the `Priority List` layout of the Priority Score model, fills the signals, scores every account, routes it to a team,
and saves the **processed data** to the local data folder.

**Part 2** is [`../build_dashboard/build_dashboard.py`](../build_dashboard/README.md): it reads the processed JSON saved here and
builds the Account Priority Dashboard in the same local data folder.

```
 EXTRACT ─► STANDARDIZE ─► MATCH ─► MERGE ─► CONFORM ─► DQ ─► SIGNALS ─► SCORE ─► ROUTE ─► DQ ─► DASHBOARD ─► LOAD
 (sources)  (per source)   (HS↔MS)  (rules)  (schema)        (mock*)    (model)  (teams)        (JSON)
```
\* All signals are **synthetic (mock)** for now, except seat growth (real, MyStaff) and fit (real, Client Propensity).

| Stage | Function | What it does |
|---|---|---|
| 1. Extract | `extract_hubspot()`, `extract_mystaff()` | Pull raw records and rename to the **source contract**. Today: Excel exports. Production: DB / API |
| 2. Standardize | `standardize_hubspot()`, `standardize_mystaff()` | Clean, filter, and map each source to one shared **staging schema** (industry, band, region, seat growth…) |
| 3. Match | `match_sources()` | Find HubSpot companies that are already MyStaff clients (CRM ID, then website domain) |
| 4. Merge | `merge_sources()` | Overlaps → one Organic Growth row (survivorship rules); the rest stay New Logo |
| 5. Conform | `conform_to_output()` | Build the output in the Priority List layout, with column groups |
| 6. DQ: merge | `run_dq_checks()` | Fail loudly if the merge breaks an invariant |
| 7. Signals | `add_signals()` | Fill the enrichment columns: **MOCK** signals + industry trend (seat growth is already real) |
| 7b. Propensity | `add_propensity()` | Fit scores from the **Client Propensity** project (seat milestones, department expansion, new-client growth): **real** from its extract or **mock**, per `PROPENSITY_MODE`. Their bands feed the Fit points (stage 8) |
| 8. Score | `score()` | Priority score, tier and Act now bonus, using the live **`dashboard_v3`** settings (`SCORING`): `dashboard_v2` plus **Fit** points (25 Organic Growth, 20 New Logo) |
| 9. Route | `route()` | Owner team (BDM / Marketing / SDR): by **Act now trigger** where there is one, otherwise a **random** draw (mock) or deterministic rules; rank within each team × goal list |
| 10. DQ: scoring | `run_scoring_checks()` | Fail loudly if scoring or routing breaks a rule |
| 11. Dashboard data | `build_dashboard_data()` | Shape the scored accounts into the Account Priority Dashboard's JSON (rows, settings, meta, QA) |
| 12. Load | `load_output()`, `load_dashboard()` | Save the processed data locally (outside the repo): the Excel output and the dashboard JSON that `build_dashboard/` reads |

`run_pipeline()` at the bottom runs stages 1–12 in one call.

**Output column groups** (`COLUMN_GROUPS`):
1. **company**: populated by this pipeline
2. **enrichment**: raw signals. **Synthetic (mock)** except `seats_added_6m`; see `enrichment_source`
3. **scoring**: derived by the scoring model, including `assigned_team`
4. **lineage**: where each value came from (keys, raw values, match info), for audit and debugging
5. **propensity**: fit scores from the Client Propensity project; their bands feed the Fit points of `priority_score` (`dashboard_v3`); see `propensity_source`

Columns added on top of the mock layout (seat growth, Act now bonus, fit) are listed in `DASHBOARD_EXTENSIONS`.
Business rules and assumptions: [`BUSINESS_RULES.md`](BUSINESS_RULES.md). How the code is built and run: [`PIPELINE.md`](PIPELINE.md).
Cells marked `# INSPECT` or `# TEST` are for looking at / checking the data; the pipeline doesn't need them.

## 0. Setup

In [1]:
from pathlib import Path
import json
import math
import random
import re
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 220)

PIPELINE_VERSION = "0.4"
RUN_DATE = pd.Timestamp.today().normalize()

## 0a. Config: sources, contracts, output schema
Swapping a source from Excel to a DB/API only needs a new branch in its extractor that returns the
same **contract** fields. Nothing downstream changes.

In [2]:
# ---- Sources -------------------------------------------------------------------------
SOURCE_MODE = "excel"          # "excel" today; "db" / "api" to be implemented in the extractors

DATA_DIR = Path.home() / "Desktop/Projects/Marketing Intelligence Priority/Data"   # outside the repo (confidential)
SOURCES = {
    "hubspot": {"path": max(DATA_DIR.glob("hubspot-core-report-*.xlsx"), key=lambda p: p.stat().st_mtime), "sheet": 0},
    "mystaff": {"path": DATA_DIR / "CONFIDENTIAL Mystaff 11092026 - Copy.xlsx", "sheet": "MyStaff"},
}
OUTPUT_PATH = DATA_DIR / "priority_company_info_combined.xlsx"
SAVE_OUTPUT = False

for name, cfg in SOURCES.items():
    print(f"{name:8} {'OK' if cfg['path'].exists() else 'MISSING'}  {cfg['path'].name}")

hubspot  OK  hubspot-core-report-artifact-1794227762851-2026-11-09.xlsx
mystaff  OK  CONFIDENTIAL Mystaff 11092026 - Copy.xlsx


In [3]:
# ---- Source contracts: source field name -> pipeline field name ------------------------
HUBSPOT_CONTRACT = {
    "Record ID":           "hs_record_id",
    "Company name":        "company_name",
    "Website URL":         "website_url",
    "Number of Employees": "employee_count",
    "Industry":            "industry_raw",
    "Annual Revenue":      "annual_revenue",
    "Financial Reports":   "financial_reports_text",   # free text; used only as region evidence
}
MYSTAFF_CONTRACT = {
    "Shortcode":              "shortcode",
    "Client":                 "company_name",
    "Active":                 "active",
    "CRM Company ID":         "crm_company_id",        # = HubSpot Record ID
    "Website URL":            "website_url",
    "Office Address State":   "office_state",
    "Office Address Country": "office_country",
    "Industry Hubspot":       "industry_hubspot",      # same taxonomy as HubSpot 'Industry'
    "Industry Roll Up":       "industry_rollup",       # Cloudstaff grouping (fallback)
    "Total Staff":            "cs_staff_current",      # Cloudstaff seats, NOT company size
    "Date of First Staff Engagement": "client_since",  # first staff start: time as a client
    "Tenure":                 "client_tenure_band",    # MyStaff's own band, e.g. "4. 13-18 months" (used when the date is blank)
    # staff added per month (gross adds), summed into seats_added_6m (seat growth signal)
    "Month to Date Movement Add": "seats_added_mtd",
    "Last Month Movement Add":    "seats_added_m1",
    "Month-2 Movement Add":       "seats_added_m2",
    "Month-3 Movement Add":       "seats_added_m3",
    "Month-4 Movement Add":       "seats_added_m4",
    "Month-5 Movement Add":       "seats_added_m5",
}
# Seat growth window: current month to date + the 5 previous months (~ last 6 months)
SEAT_WINDOW = ["seats_added_mtd", "seats_added_m1", "seats_added_m2", "seats_added_m3", "seats_added_m4", "seats_added_m5"]

In [4]:
# ---- Output schema (mirrors the mock 'Priority List') --------------------------------
GOALS = {"hubspot": "New Logo", "mystaff": "Organic Growth"}

COMPANY_COLS = ["account_id", "company_name", "goal", "industry", "employee_band", "region"]

ENRICHMENT_COLS = [            # raw signals (stage 7)
    "web_visits_30d", "team_builder_visit", "form_fill_30d",
    "news_signal", "news_signal_date", "job_postings_30d", "hiring_spike",
    "diamond_referral", "prior_bpo_flag",       # prior_bpo_flag is shown as "Competitor client" on the dashboard
    "industry_trend",
    "seats_added_6m",                           # dashboard extension: seats added in ~6 months (Organic Growth)
]
SCORING_COLS = [               # derived by the scoring model from company + enrichment columns
    "industry_modifier", "web_signal (0-1)", "news_jobs_signal (0-1)",
    "web_pts", "news_jobs_pts", "diamond_pts", "bpo_pts", "base_points", "priority_score",
    "estimated_fields", "data_coverage", "hot_trigger", "tier", "assigned_team", "rank_in_team_list", "sort_key",
    "seat_signal (0-1)", "seat_pts", "score_before_bonus", "act_now_bonus", "act_now_triggers",   # dashboard extensions
    "fit_signal (0-1)", "fit_pts",                                       # dashboard_v3: Fit component (Client Propensity)
]
DASHBOARD_EXTENSIONS = ["seats_added_6m", "seat_signal (0-1)", "seat_pts", "score_before_bonus", "act_now_bonus",
                        "act_now_triggers", "fit_signal (0-1)", "fit_pts"]
REAL_SIGNAL_COLS = ["seats_added_6m"]          # signals that already come from a real source (MyStaff)
LINEAGE_COLS = [               # not in the mock; traceability for every row
    "source", "shortcode", "hubspot_record_id", "crm_company_id", "match_type", "hs_company_name",
    "website_domain", "industry_sector", "industry_raw", "industry_source", "employee_count", "employee_source",
    "region_source", "annual_revenue", "cs_staff_current", "client_since", "client_tenure_band",
    "name_match_review", "note", "enrichment_source",
]
PROPENSITY_COLS = [            # fit scores from the Client Propensity project (stage 7b, PROPENSITY_MODE). Feed the Fit points (dashboard_v3)
    # Organic Growth clients (joined on account_id). Seat milestone = the client's next step: 1-4 seats -> 5, 5-9 -> 10,
    # 10-19 -> 20 (20+ out of scope); _p = chance of reaching it within a year
    "fit_seat_milestone", "fit_seat_milestone_p", "fit_seat_milestone_band", "fit_seat_milestone_why",
    "fit_dept_expand_p", "fit_dept_expand_band", "fit_dept_vs_peers", "fit_dept_why",
    # departments so far; one-department clients: chance of a second department, banded among one-department clients
    "depts_so_far", "depts_so_far_names", "fit_second_dept_p", "fit_second_dept_band",
    "likely_next_dept", "likely_next_dept_by_lift", "seats_change_6m_net",
    # young Organic Growth clients, and New Logo prospects (joined on industry|country)
    "fit_growth_p", "fit_growth_band", "fit_growth_why", "fit_reach_10_p", "fit_reach_10_band", "fit_reach_10_why",
    # New Logo prospects only
    "fit_branch_1y_p", "fit_branch_1y_band", "fit_branch_1y_why", "likely_first_new_dept", "fit_outlook",
    # every row
    "fit_coverage", "propensity_source",
]
DASHBOARD_EXTENSIONS += PROPENSITY_COLS
OUTPUT_COLS = COMPANY_COLS + ENRICHMENT_COLS + SCORING_COLS + LINEAGE_COLS + PROPENSITY_COLS
COLUMN_GROUPS = {
    **{c: "1_company" for c in COMPANY_COLS},
    **{c: "2_enrichment" for c in ENRICHMENT_COLS},
    **{c: "3_scoring" for c in SCORING_COLS},
    **{c: "4_lineage" for c in LINEAGE_COLS},
    **{c: "5_propensity" for c in PROPENSITY_COLS},
}
GROUP_DESCRIPTION = {
    "1_company":    "Populated by this pipeline (merge + formatting)",
    "2_enrichment": "Raw signals. MOCK (synthetic) while ENRICHMENT_MODE = 'mock', except seats_added_6m (real, MyStaff)",
    "3_scoring":    "Derived by the scoring model (dashboard-adjusted settings) from company + enrichment columns, plus routing",
    "4_lineage":    "Not in the mock; industry_sector (parent of industry) plus source keys, raw values and match info",
    "5_propensity": "Fit scores from the Client Propensity project: how likely clients / prospects like this are to add seats or departments. REAL (its extract) or MOCK per PROPENSITY_MODE; see propensity_source. Their bands feed the Fit points of priority_score (dashboard_v3)",
}

# Shared staging schema: what each standardize_* returns
STAGING_COLS = [
    "source", "shortcode", "hubspot_record_id", "crm_company_id", "company_name", "name_key", "website_domain",
    "industry_raw", "industry", "industry_source", "employee_count", "employee_band", "employee_source",
    "region", "region_source", "annual_revenue", "cs_staff_current", "seats_added_6m",
    "client_since", "client_tenure_band",          # MyStaff only: first staff start date, MyStaff tenure band
]

# Allowed values for bands and regions. Industry uses our own taxonomy (INDUSTRY_GROUPS in 0b).
REF_BANDS = ["1-10", "11-50", "51-200", "201-500", "500+"]
REF_REGIONS = ["AU-NSW", "AU-QLD", "AU-SA", "AU-VIC", "AU-WA", "NZ", "UK", "US"]

In [5]:
# INSPECT: the output schema: every column listed once, in exactly one group
assert len(OUTPUT_COLS) == len(set(OUTPUT_COLS)), "a column is listed twice"
assert set(COLUMN_GROUPS) == set(OUTPUT_COLS), "a column has no group"
assert set(DASHBOARD_EXTENSIONS) <= set(OUTPUT_COLS) and set(REAL_SIGNAL_COLS) <= set(ENRICHMENT_COLS)
print(f"{len(OUTPUT_COLS)} output columns, including {len(DASHBOARD_EXTENSIONS)} dashboard extensions: {DASHBOARD_EXTENSIONS}")
pd.Series(COLUMN_GROUPS).groupby(lambda c: COLUMN_GROUPS[c]).size().to_frame("n_columns").assign(
    description=lambda d: d.index.map(GROUP_DESCRIPTION))

56 output columns, including 6 dashboard extensions: ['seats_added_6m', 'seat_signal (0-1)', 'seat_pts', 'score_before_bonus', 'act_now_bonus', 'act_now_triggers']


,n_columns,description
1_company,6,Populated by this pipeline (merge + formatting)
2_enrichment,11,Raw signals. MOCK (synthetic) while ENRICHMENT_MODE = 'm...
3_scoring,21,Derived by the scoring model (dashboard-adjusted setting...
4_lineage,18,Not in the mock; industry_sector (parent of industry) pl...


## 0b. Business rules: mappings and thresholds
Every rule the pipeline applies lives here, as data. See `BUSINESS_RULES.md` for the reasoning behind each one.

In [6]:
# ---- MyStaff row filters ----------------------------------------------------------------
TEST_ACCOUNT_PATTERN = r"^test|\bqa\b"        # client names that are QA / test accounts
PLACEHOLDER_ROLLUP = {"CLOUDSTAFF", "Test Record - No Category", "Cloudstaff Test Account",
                      "-", "No Info Available", "Others"}   # real clients, but the industry label is a placeholder

In [7]:
# ---- Industry taxonomy: 18 industry groups, each under one of 6 sectors -------------------------
# Built from the values actually present in HubSpot 'Industry' and MyStaff 'Industry Hubspot'
# (both use HubSpot's industry list). Not the mock's 12 industries. See BUSINESS_RULES.md §6.
INDUSTRY_GROUPS = {
    "Technology": ["Information Technology and Services", "Computer Software", "Software Development", "Information Services",
                   "Information Technology", "Internet", "Computer Networking", "Computer & Network Security",
                   "Computer Hardware", "Telecommunications"],
    "Real Estate": ["Real Estate", "Commercial Real Estate"],
    "Accounting": ["Accounting"],
    "Construction & Engineering": ["Construction", "Building Materials", "Civil Engineering", "Architecture & Planning",
                                   "Mechanical or Industrial Engineering"],
    "Financial Services": ["Financial Services", "Banking", "Capital Markets", "Investment Management",
                           "Venture Capital & Private Equity", "Mortgage Broker"],
    "Insurance": ["Insurance"],
    "Legal": ["Legal Services", "Law Practice"],
    "Professional Services": ["Management Consulting", "Human Resources", "Staffing and Recruiting", "Executive Office",
                              "Translation and Localization", "Market Research", "Research", "Think Tanks"],
    "Marketing, Media & Creative": ["Marketing and Advertising", "Online Media", "Media Production", "Publishing", "Printing",
                                    "Graphic Design", "Design", "Photography", "Broadcast Media", "Entertainment",
                                    "Performing Arts", "Arts and Crafts"],
    "Healthcare & Wellness": ["Hospital & Health Care", "Medical Practice", "Medical Devices", "Mental Health Care",
                              "Pharmaceuticals", "Alternative Medicine", "Veterinary", "Health, Wellness and Fitness"],
    "Education & Training": ["Education Management", "E-Learning", "Higher Education", "Primary/Secondary Education",
                             "Professional Training & Coaching"],
    "Retail & Consumer Goods": ["Retail", "Ecommerce", "Wholesale", "Apparel & Fashion", "Textiles", "Consumer Goods",
                                "Consumer Electronics", "Cosmetics", "Luxury Goods & Jewelry", "Sporting Goods",
                                "Food & Beverages", "Wine and Spirits", "Food Production", "Ranching"],
    "Hospitality, Travel & Leisure": ["Hospitality", "Leisure, Travel & Tourism", "Restaurants", "Events Services", "Sports"],
    "Business & Consumer Services": ["Facilities Services", "Security and Investigations", "Consumer Services",
                                     "Individual & Family Services"],
    "Manufacturing & Industrial": ["Machinery", "Industrial Automation", "Electrical/Electronic Manufacturing",
                                   "Motor Vehicle Manufacturing", "Automotive", "Chemicals", "Packaging and Containers",
                                   "Mining & Metals"],
    "Transport & Logistics": ["Logistics and Supply Chain", "Transportation/Trucking/Railroad", "Package/Freight Delivery",
                              "Maritime", "Airlines/Aviation", "Import and Export"],
    "Energy & Environment": ["Oil & Energy", "Utilities", "Renewables & Environment", "Environmental Services"],
    "Non-profit & Public Sector": ["Non-Profit Organization Management", "Civic & Social Organization", "Public Safety"],
}
INDUSTRY_SECTOR = {
    "Technology": "Technology & Media", "Marketing, Media & Creative": "Technology & Media",
    "Real Estate": "Property & Construction", "Construction & Engineering": "Property & Construction",
    "Accounting": "Financial & Professional Services", "Financial Services": "Financial & Professional Services",
    "Insurance": "Financial & Professional Services", "Legal": "Financial & Professional Services",
    "Professional Services": "Financial & Professional Services",
    "Retail & Consumer Goods": "Consumer, Hospitality & Services", "Hospitality, Travel & Leisure": "Consumer, Hospitality & Services",
    "Business & Consumer Services": "Consumer, Hospitality & Services", "Non-profit & Public Sector": "Consumer, Hospitality & Services",
    "Healthcare & Wellness": "Health & Education", "Education & Training": "Health & Education",
    "Manufacturing & Industrial": "Industrial, Logistics & Energy", "Transport & Logistics": "Industrial, Logistics & Energy",
    "Energy & Environment": "Industrial, Logistics & Energy",
}
# Values that mean "unknown" -> industry left blank.
# Outsourcing/Offshoring: 26 of 27 such MyStaff clients have Roll Up = CLOUDSTAFF, i.e. Cloudstaff's own industry stamped by default.
UNCLASSIFIED_INDUSTRY = {"-", "Other", "Outsourcing/Offshoring"}

INDUSTRY_MAP = {raw: group for group, raws in INDUSTRY_GROUPS.items() for raw in raws}   # raw value -> group

# ---- MyStaff fallback: Cloudstaff 'Industry Roll Up' -> industry group
#      (used only when 'Industry Hubspot' is blank / unclassified; placeholder roll-ups stay blank)
ROLLUP_MAP = {
    "Advanced Technology Services": "Technology", "Software & Technology Development": "Technology",
    "Telecommunications & Information Technology": "Technology", "Information Technology": "Technology", "Internet": "Technology",
    "Real Estate Services": "Real Estate", "Real Estate": "Real Estate",
    "Accounting Services": "Accounting",
    "Comprehensive Financial Services": "Financial Services", "Financial Services": "Financial Services",
    "Construction & Infrastructure": "Construction & Engineering", "Specialized Engineering & Technical Services": "Construction & Engineering",
    "Professional Advisory": "Professional Services", "Business Operations & Enterprise Solutions": "Professional Services",
    "Professional Services": "Professional Services", "Research, Development & Innovation": "Professional Services",
    "Human Resources": "Professional Services",
    "Legal & Security Services": "Legal", "Legal Services": "Legal",
    "Digital Marketing & Advertising": "Marketing, Media & Creative", "Marketing & Advertising": "Marketing, Media & Creative",
    "Marketing Services": "Marketing, Media & Creative", "Creative and Digital Media": "Marketing, Media & Creative",
    "Design, Arts & Entertainment": "Marketing, Media & Creative", "Media & Entertainment": "Marketing, Media & Creative",
    "Publishing, Editing & Literature": "Marketing, Media & Creative",
    "Health & Welfare Services": "Healthcare & Wellness", "Healthcare & Medical Services": "Healthcare & Wellness",
    "Healthcare Services": "Healthcare & Wellness", "Health & Wellness Services": "Healthcare & Wellness",
    "Pharmaceuticals & Life Sciences": "Healthcare & Wellness", "Health & Safety Products": "Healthcare & Wellness",
    "Education & Knowledge Services": "Education & Training", "Professional Coaching & Development": "Education & Training",
    "Retail & Commerce": "Retail & Consumer Goods", "Retail & Consumer Services": "Retail & Consumer Goods",
    "Retail & Consumer Goods": "Retail & Consumer Goods", "Apparel, Fashion & Textiles": "Retail & Consumer Goods",
    "Consumer Products & Electronics": "Retail & Consumer Goods", "Wholesale & Distribution": "Retail & Consumer Goods",
    "Food, Beverage & Agriculture": "Retail & Consumer Goods", "Food & Beverage": "Retail & Consumer Goods",
    "Hospitality, Travel & Leisure": "Hospitality, Travel & Leisure", "Event Management & Services": "Hospitality, Travel & Leisure",
    "Sports, Fitness & Recreation": "Hospitality, Travel & Leisure", "Arts, Entertainment & Recreation": "Hospitality, Travel & Leisure",
    "Personal Care Products & Services": "Business & Consumer Services", "Consumer Services": "Business & Consumer Services",
    "Facilities Management": "Business & Consumer Services",
    "Manufacturing & Industrial Products": "Manufacturing & Industrial", "Manufacturing & Industrial Services": "Manufacturing & Industrial",
    "Automotive & Motor Vehicles": "Manufacturing & Industrial", "Raw Material Extraction & Processing": "Manufacturing & Industrial",
    "Transportation & Logistics": "Transport & Logistics", "Transportation & Logistics Services": "Transport & Logistics",
    "Energy & Environmental Services": "Energy & Environment", "Environmental Services": "Energy & Environment",
    "Social & Civic Organizations": "Non-profit & Public Sector", "Non-Profit Organizations": "Non-profit & Public Sector",
}

assert len(INDUSTRY_MAP) == sum(map(len, INDUSTRY_GROUPS.values())), "a raw value is in two industry groups"
assert not set(INDUSTRY_MAP) & UNCLASSIFIED_INDUSTRY
assert set(INDUSTRY_GROUPS) == set(INDUSTRY_SECTOR)
assert set(ROLLUP_MAP.values()) <= set(INDUSTRY_GROUPS)

In [8]:
# ---- Employee band (upper bounds, inclusive) ------------------------------------------------
BAND_EDGES = [(10, "1-10"), (50, "11-50"), (200, "51-200"), (500, "201-500"), (np.inf, "500+")]

# ---- Region: MyStaff office address ------------------------------------------------------
COUNTRY_CODE = {
    "australia": "AU", "united states of america": "US", "united states": "US", "usa": "US",
    "united kingdom of great britain and northern ireland": "UK", "united kingdom": "UK", "jersey": "UK",
    "new zealand": "NZ", "canada": "CA", "singapore": "SG", "hong kong": "HK", "ireland": "IE",
    "united arab emirates": "AE", "philippines": "PH",
}
AU_STATE = {
    "new south wales": "NSW", "nsw": "NSW", "victoria": "VIC", "vic": "VIC", "queensland": "QLD", "qld": "QLD",
    "south australia": "SA", "sa": "SA", "western australia": "WA", "wa": "WA",
    "australian capital territory": "ACT", "act": "ACT", "tasmania": "TAS", "tas": "TAS",
    "northern territory": "NT", "nt": "NT",
}

# ---- Region: HubSpot evidence hierarchy (first hit wins) ------------------------------------
TLD_COUNTRY = {"au": "AU", "nz": "NZ", "uk": "UK", "us": "US", "ca": "CA", "ie": "IE", "sg": "SG"}   # 1. country-code domain
NAME_COUNTRY = [                                                                                  # 2. company name
    (r"\bpte\.?\s*(ltd|limited)\b", "SG"),
    (r"\bpty\b", "AU"),
    (r"\baustralia\b", "AU"),
    (r"\bnew zealand\b|\bnz\b", "NZ"),
    (r"\bunited kingdom\b|\buk\b", "UK"),
    (r"\bcanada\b", "CA"),
    (r"\bsingapore\b", "SG"),
]
REPORT_COUNTRY = {                                                                                # 3. financial reports text
    "AU": r"\basic\b|\babn\b|\bacn\b|\.gov\.au|\.com\.au|australia|abr\.business|au\.companiesdb|\bpty\b",
    "NZ": r"new zealand|\.govt\.nz|\.co\.nz|\bnzbn\b",
    "UK": r"companies house|\.gov\.uk|\.co\.uk|united kingdom|\bengland\b|\bfca\b",
    "CA": r"canada|\.gc\.ca",
    "SG": r"singapore|\bacra\b",
}
REPORT_US = r"\bsec\b|edgar|united states|\bdelaware\b|\bu\.s\.|\busa\b"
NAME_NON_US = r"\b(ltd|limited|gmbh|plc|bv|ag|ehf|aps)\b\.?\)?\s*$"                               # 4. non-US, country unclear -> blank
ASSUMED_TLD = {"com": "US"}                                                                       # 5. PROTOTYPE ASSUMPTION (last resort)

# ---- Matching ------------------------------------------------------------------------------
MATCH_RANK = {"CRM ID + domain": 0, "CRM ID": 1, "Website domain": 2, "Website domain (name tie-break)": 3}
AMBIGUOUS = "Website domain (ambiguous)"

In [9]:
# ---- Enrichment ------------------------------------------------------------------------------
# "mock": synthetic signals made with the mock workbook's generator rules
#         (reference materials/scripts/generate_mock_workbook.py). FICTIONAL: for building and testing only.
#         Every row is tagged enrichment_source = "MOCK ..." and the saved file gets a _MOCK_SIGNALS suffix.
# "live": real sources (not built yet).
ENRICHMENT_MODE = "mock"
MOCK_SEED = 42
SCORING_AS_OF = RUN_DATE                       # news recency is measured from this date

# ---- Propensity signals (stage 7b): fit scores from the Client Propensity project -------------------------------
# "real": the newest Client Propensity extract in DATA_DIR (Client Propensity/run_pipeline.py copies it here). Organic
#         Growth rows join on account_id; New Logo rows on industry|country (its prospect profiles).
# "mock": SYNTHETIC fit_* values, seeded per account_id (MOCK_SEED). FICTIONAL: for building and testing the page.
#         Rows are tagged propensity_source = "MOCK ..." and the saved files get the _MOCK_SIGNALS suffix.
# "off":  the propensity columns stay blank.
PROPENSITY_MODE = "real"
PROPENSITY_ACCOUNTS_GLOB = "*_Propensity_Extract_propensity_accounts.csv"   # names start with the run date: newest = last
PROPENSITY_MIN_MATCH = 0.95      # DQ warning below this share of Organic Growth rows found in the extract
PROPENSITY_MIN_SCORED = 0.70     # DQ warning below this share of Organic Growth rows with a fit score
PROPENSITY_MAX_AGE_DAYS = 45     # DQ warning when the extract's as-of date is further than this from SCORING_AS_OF
# Region -> the country level the propensity models use: AU* = Australia, these four, blank = Unknown, anything else = Other
PROPENSITY_COUNTRY = {"US": "United States", "UK": "United Kingdom", "NZ": "New Zealand", "CA": "Canada"}

def is_mock_run():
    """True when any signal is synthetic: saved files then get the _MOCK_SIGNALS suffix."""
    return ENRICHMENT_MODE == "mock" or PROPENSITY_MODE == "mock"

# ---- Scoring settings: four profiles ---------------------------------------------------------------
# "workbook":     the mock workbook's 'Settings' sheet. Kept as a regression test (must reproduce accounts.csv).
# "dashboard":    the dashboard's adjusted settings (work/account_priority_dashboard/score_adjustments.json):
#                 round points, 25% baselines, seat growth (Organic Growth) and an Act now bonus. Live until v0.2.
# "dashboard_v2": "dashboard" plus the 2026-11-10 scoring review (BUSINESS_RULES.md §11.2). Live until v0.3.
# "dashboard_v3": "dashboard_v2" plus a Fit component from the Client Propensity scores (2026-11-13). THE LIVE SETTINGS.
_HOT = {"diamond": True, "form_fill": True, "team_builder": True, "top_news": True,
        "top_news_min_strength": 1.0, "top_news_days": 30,
        # extended triggers: OFF in the workbook and reference profiles, ON in the live dashboard profile (_HOT_LIVE)
        "team_builder_prospect": False,            # Team Builder visit by a prospect (New Logo)
        "hiring_spike": False, "hiring_min_postings": 18,   # hiring spike, or 18+ job postings (New Logo)
        "heavy_visits_min": None,                  # N+ website visits in 30 days with no form fill (both goals)
        "campaign_news": [], "campaign_news_days": None}   # news categories that are Marketing triggers, and their window in days
_HOT_LIVE = {**_HOT, "team_builder_prospect": True, "hiring_spike": True, "heavy_visits_min": 10,
             "campaign_news": ["Restructuring", "Leadership change"], "campaign_news_days": 60}
_TIER = {"tier1": 70, "tier2": 40, "cap_tier1_on_coverage": False, "tier1_min_coverage": 0.6}
_MODIFIER = {"at_trend_0": 0.9, "at_trend_1": 1.1}

SCORING_WORKBOOK = {
    "max_points": {"New Logo":       {"web": 35, "news": 35, "diamond": 15, "bpo": 15, "seat": 0},
                   "Organic Growth": {"web": 50, "news": 50, "diamond": 0,  "bpo": 0,  "seat": 0}},
    "baseline":   {"web": 0.35, "news": 0.30, "seat": 0.0},    # share of max used when a component is unknown
    "web":        {"tb_share": 0.4, "ff_share": 0.3, "vol_share": 0.3, "vol_cap": 20},
    "news":       {"news_share": 0.6, "jobs_share": 0.4, "jobs_cap": 20},
    "seat_bands": [],
    "modifier":   _MODIFIER, "tier": _TIER, "hot": _HOT,
    "act_now_bonus": None,
    "round_signals": 3,                                       # the workbook rounds each 0-1 signal to 3 decimals
}
SCORING_DASHBOARD = {
    "max_points": {"New Logo":       {"web": 40, "news": 40, "diamond": 10, "bpo": 10, "seat": 0},     # bpo = Competitor client
                   "Organic Growth": {"web": 40, "news": 40, "diamond": 0,  "bpo": 0,  "seat": 20}},
    "baseline":   {"web": 0.25, "news": 0.25, "seat": 0.25},
    "web":        {"tb_share": 0.5, "ff_share": 0.25, "vol_share": 0.25, "vol_cap": 20},
    "news":       {"news_share": 0.625, "jobs_share": 0.375, "jobs_cap": 20},
    "seat_bands": [(0, 0.0), (1, 0.25), (6, 0.5), (11, 1.0)],  # seats added in ~6 months (from) -> share of seat max
    "modifier":   _MODIFIER, "tier": _TIER, "hot": _HOT_LIVE,
    "act_now_bonus": {"default": 15, "Diamond referral": 7.5},   # every trigger; added after the industry nudge; Diamond already earns flag points
    "round_signals": None,
}
# The dashboard profile WITHOUT the extended triggers: what the mock dashboard's prep.py produced. Used by golden test 2 only.
SCORING_DASHBOARD_REFERENCE = {**SCORING_DASHBOARD, "hot": _HOT}
# dashboard_v2 (live until v0.3): the scoring review of 2026-11-10. Same points and signals as "dashboard", plus:
#   * rarer Act now triggers: heavy website visits need 20+ visits (was 10), restructuring / leadership news 30 days (was 60)
#   * every trigger adds to the bonus, not only the first: the best bonus per trigger family (TRIGGER_FAMILY), the families
#     sorted by bonus, and each further family worth stack_decay x the one before (15, then +7.5, then +3.75 ...)
#   * an unknown component is scored at the AVERAGE signal of the same goal's accounts where it is known (not a flat 25%)
#   * tiers by rank within each goal (top 10% Tier 1, next 30% Tier 2), not fixed scores
#   * sort_key = score: the bonus moves Act now accounts up, but they are no longer pinned above every other account
# Profiles that leave these keys out keep the old behaviour (flat baseline, first trigger only, score tiers, +1000 sort).
_HOT_V2 = {**_HOT_LIVE, "heavy_visits_min": 20, "campaign_news_days": 30}
SCORING_DASHBOARD_V2 = {
    **SCORING_DASHBOARD,
    "hot": _HOT_V2,
    "unknown_fill": "known_mean",          # "known_mean" or "baseline"; "baseline" (25%) stays the fallback when no account of a goal has the signal
    "act_now_bonus": {"default": 15, "Diamond referral": 7.5, "stack_decay": 0.5},
    "tier": {"method": "percentile", "tier1_share": 0.10, "tier2_share": 0.30},   # share of each goal's accounts; ties share the higher tier
    "hot_first": False,                    # True = the workbook's sort_key (+1000 for any Act now trigger)
}
# dashboard_v3 (LIVE, 2026-11-13): dashboard_v2 plus a FIT component from the Client Propensity scores (stage 7b), rolled
# into priority_score like the other signals. Points: Organic Growth fit 25 (website 30, news/jobs 30, seat growth 15),
# New Logo fit 20 (website 30, news/jobs 30, Diamond 10, Competitor client 10). Seat growth drops 20 -> 15 because the
# seat-milestone fit already reuses recent seat adds (rank correlation 0.53 with seats_added_6m).
# The 0-1 fit signal blends the account's propensity bands (band_value):
#   Organic Growth: seat part x seat_weight + department part x dept_weight. Seat part = the band of the client's NEXT seat
#     milestone (1-4 seats -> 5+, 5-9 -> 10+, 10-19 -> 20+; 20+ seats has none). Department part = fit_second_dept_band, for
#     ONE-department clients only (banded among them): the goal is clients that start with one department and grow past it.
#     Clients with 2+ departments score multi_dept_value (0) on the department part, with the weights kept, so their fit
#     is at most seat_weight (15 of 25 points) and one-department clients likely to add a second rank above them. 2+
#     departments with no seat milestone (20+ seats) = fit unknown (the goal's average).
#   New Logo (industry|country profile, "if we sign them"): growth part x growth_weight + department part x dept_weight.
#     Growth part = the better of the 5-seat (1 year) and 10-seat (2 years) bands; department part = fit_branch_1y_band.
#   A missing part leaves the other on its own. No fit at all (e.g. a client with no current seats) = an unknown component:
#   the average fit of the same goal's accounts (unknown_fill), listed in estimated_fields as "Fit".
FIT = {
    "band_value": {"High": 1.0, "Medium": 0.5, "Low": 0.0},
    "Organic Growth": {"seat_weight": 0.6, "dept_weight": 0.4, "multi_dept_value": 0.0},
    "New Logo": {"growth_weight": 0.75, "dept_weight": 0.25,   # 15 + 5 of the 20 fit points (2026-11-13; was 0.7 / 0.3)
                 # the "Likely 5+ seats" chip compares a prospect with the AVERAGE PROSPECT on the list ("prospects"), not with past
                 # new clients ("clients"): prospects are scored as an unknown role launching with 1-2 staff, so they average 5.7%
                 # against 9.6% for past new clients, and none would reach 1.5x the latter
                 "growth_compare": "prospects"},
    "chip_band": "High",                   # a fit chip (Why column) shows when that score's band is this ...
    "chip_min_lift": 1.5,                  # ... AND its chance is at least this many times the usual rate (the score's
                                           # historical base rate). Bands are relative: the top third of 1-4 seat clients
                                           # starts at 5%, below the usual 7%, and New Logo profiles barely differ from average
}
SCORING_DASHBOARD_V3 = {
    **SCORING_DASHBOARD_V2,
    "max_points": {"New Logo":       {"web": 30, "news": 30, "diamond": 10, "bpo": 10, "seat": 0,  "fit": 20},
                   "Organic Growth": {"web": 30, "news": 30, "diamond": 0,  "bpo": 0,  "seat": 15, "fit": 25}},
    "baseline": {**SCORING_DASHBOARD_V2["baseline"], "fit": 0.25},   # fallback only (unknown_fill = known_mean)
    # News and jobs: news and hiring 15 points each of the 30 (2026-11-13; dashboard / dashboard_v2: 0.625 / 0.375 = 18.75 / 11.25)
    "news": {**SCORING_DASHBOARD_V2["news"], "news_share": 0.5, "jobs_share": 0.5},
    "fit": FIT,
}
assert all(abs(sum(w for k, w in v.items() if k.endswith("_weight")) - 1) < 1e-9 for g, v in FIT.items()
           if g in ("Organic Growth", "New Logo")), "fit weights must add up to 1"
ACT_NOW_TARGET_SHARE = 0.10                # DQ warning when more than this share of a goal's accounts has an Act now trigger
SCORING_PROFILES = {"workbook": SCORING_WORKBOOK, "dashboard_reference": SCORING_DASHBOARD_REFERENCE,
                    "dashboard": SCORING_DASHBOARD, "dashboard_v2": SCORING_DASHBOARD_V2, "dashboard_v3": SCORING_DASHBOARD_V3}
SCORING_PROFILE = "dashboard_v3"
SCORING = SCORING_PROFILES[SCORING_PROFILE]
for name, prof in SCORING_PROFILES.items():
    assert all(sum(v.values()) == 100 for v in prof["max_points"].values()), f"{name}: max points must total 100 per goal"
NEWS_STRENGTH = {                              # order matters for the mock generator
    "Funding round": 1.0, "PE buyout": 1.0, "M&A": 0.8, "Expansion / new office": 0.7,
    "Restructuring": 0.5, "Leadership change": 0.3, "Other / Unclassified": 0.3, "None": 0.0,
}
NEWS_UNMAPPED = "Other / Unclassified"         # strength used for a category not in the table
RECENCY = [(0, 1.0), (31, 0.7), (91, 0.4), (181, 0.0)]   # (days since news, from) -> factor

# Industry trend (0-1) per industry group: MOCK values carried over from the workbook's 12 industries where a group
# has a clear equivalent. Groups without one are left out -> modifier 1.0 (no effect), as the workbook does for unknowns.
INDUSTRY_TREND = {
    "Healthcare & Wellness": 0.8, "Transport & Logistics": 0.7, "Legal": 0.5, "Real Estate": 0.4,
    "Accounting": 0.7, "Financial Services": 0.7, "Retail & Consumer Goods": 0.9, "Construction & Engineering": 0.6,
    "Insurance": 0.5, "Marketing, Media & Creative": 0.6, "Technology": 0.8, "Education & Training": 0.3,
    "Hospitality, Travel & Leisure": 0.4,
}
assert set(INDUSTRY_TREND) <= set(INDUSTRY_GROUPS)

# ---- Routing: one owner team per account (the mock dashboard's three roles) -----------------------
TEAM_GOALS = {                                 # which goals each team may own
    "BDM":       ["New Logo", "Organic Growth"],
    "Marketing": ["New Logo", "Organic Growth"],
    "SDR":       ["New Logo"],                 # SDR owns New Logo only
}
BDM_MIN_SCORE = 60                             # the workbook's fixed 'New Logo score sent to BDM' (workbook golden test only)
# BDM candidates (2026-11-13): an account WITHOUT an Act now trigger whose score is in the top BDM_TOP_SHARE of its goal's
# accounts without a trigger (ranked like the tiers: ties share the place). Relative, so it keeps working when the score
# levels move (e.g. mock -> real signals). With the fixed 60, no New Logo account without a trigger qualified once the mock
# trigger rates were tuned (highest score 44), so BDM's New Logo list held only triggered accounts.
BDM_TOP_SHARE = 0.10

# ROUTING_MODE = "random" (live): the mock dashboard's own role assignment (work/account_priority_dashboard/make_mock_data.py):
#   a seeded random draw with set probabilities per account type. A MOCK stand-in for real owner assignment, tagged in
#   enrichment_source. "rules": deterministic rules (see route_account_rules()). Both modes keep SDR to New Logo only.
ROUTING_MODE = "random"
#   Accounts with an Act now trigger do NOT take part in the draw: they go to the trigger's team (TRIGGER_TEAM below).
ROUTING_PROBABILITIES = {                      # account type (without an Act now trigger) -> probability of each team
    "Organic Growth":          {"BDM": 0.70, "Marketing": 0.30},
    "New Logo, top 10%":       {"BDM": 0.80, "Marketing": 0.20},   # top BDM_TOP_SHARE of New Logo accounts without a trigger
    "New Logo, other":         {"SDR": 0.60, "Marketing": 0.40},
}
assert all(abs(sum(p.values()) - 1) < 1e-9 for p in ROUTING_PROBABILITIES.values()), "routing probabilities must add up to 1"
assert all(set(p) <= {t for t, goals in TEAM_GOALS.items() if g.split(",")[0] in goals}
           for g, p in ROUTING_PROBABILITIES.items()), "a team in ROUTING_PROBABILITIES may not own that goal"

# ---- Act now trigger -> team. Accounts with a trigger go to its team in BOTH routing modes -------------------
TRIGGER_TEAM = {                               # trigger -> {goal: team}; a trigger or goal not listed goes to BDM
    "Diamond referral":     {"New Logo": "BDM"},
    "Form fill":            {"New Logo": "BDM", "Organic Growth": "BDM"},
    "Team Builder visit":   {"New Logo": "SDR", "Organic Growth": "BDM"},      # a prospect's visit -> SDR, a client's -> BDM
    "Funding round":        {"New Logo": "BDM", "Organic Growth": "BDM"},
    "PE buyout":            {"New Logo": "BDM", "Organic Growth": "BDM"},
    "Hiring spike":         {"New Logo": "SDR"},                                # hiring spike or 18+ job postings
    "Heavy website visits": {"New Logo": "Marketing", "Organic Growth": "Marketing"},   # 20+ visits in 30 days (dashboard_v2), no form fill
    "Restructuring":        {"New Logo": "Marketing", "Organic Growth": "Marketing"},   # news in the last 30 days (dashboard_v2)
    "Leadership change":    {"New Logo": "Marketing", "Organic Growth": "Marketing"},
}
assert all(team in TEAM_GOALS and goal in TEAM_GOALS[team] for t in TRIGGER_TEAM.values() for goal, team in t.items()), \
    "TRIGGER_TEAM gives a team a goal it may not own"

# ---- Act now trigger -> family. Stacked bonuses count one trigger per family (they measure the same thing) --------
TRIGGER_FAMILY = {
    "Diamond referral": "Referral",
    "Form fill": "Website", "Team Builder visit": "Website", "Heavy website visits": "Website",
    "Funding round": "News", "PE buyout": "News", "Restructuring": "News", "Leadership change": "News",
    "Hiring spike": "Hiring",
}
assert set(TRIGGER_FAMILY) == set(TRIGGER_TEAM), "every trigger needs a family and a team"

# ---- Golden tests ---------------------------------------------------------------------------------
GOLDEN_PATH = Path("../reference materials/data/accounts.csv")          # 400 accounts scored by the mock workbook
DASHBOARD_GOLDEN_PATH = Path("../data/account_priority_dashboard.json")  # 800 accounts scored by the dashboard (prep.py)
GOLDEN_AS_OF = pd.Timestamp("2026-11-08")
GOLDEN_INDUSTRY_TREND = {"Healthcare": 0.8, "Logistics": 0.7, "Legal": 0.5, "Real Estate": 0.4, "Accounting & Finance": 0.7,
                         "E-commerce": 0.9, "Construction": 0.6, "Insurance": 0.5, "Marketing Agency": 0.6,
                         "SaaS / Tech": 0.8, "Education": 0.3, "Hospitality": 0.4}

In [10]:
# ---- Dashboard data (stages 11-12) --------------------------------------------------------------------
# The Account Priority Dashboard is built by ../build_dashboard/build_dashboard.py (part 2 of the pipeline).
# This notebook only saves the JSON it reads, to DATA_DIR (outside the repo).
DASHBOARD_JSON_PATH = DATA_DIR / "account_priority_dashboard_data.json"   # build_dashboard.py reads the newest account_priority_dashboard_data*.json

DASH_ROLES = ["Marketing", "SDR", "BDM"]       # the dashboard's role order
DASH_LISTS = [f"{r}|{g}" for r in DASH_ROLES for g in GOALS.values() if g in TEAM_GOALS[r]]   # 5 lists: SDR has no Organic Growth
CHIP_STRONG, CHIP_MODERATE = 0.66, 0.33        # strength dots on the Why chips: ●●● / ●●○ / ●○○ (same as prep.py)
GROWING_INDUSTRY = 0.7                         # industry_trend at or above this shows a "Growing industry" chip
MOCK_CHIP_TAG = " · mock"                      # added to every Why chip that comes from a mock signal

## 0c. Helpers

In [11]:
def validate_contract(df, contract, source):
    missing = set(contract.values()) - set(df.columns)
    if missing:
        raise ValueError(f"{source}: missing contract fields {sorted(missing)}")
    return df[list(contract.values())]

def to_domain(url):
    """'https://www.Foo.com.au/about' -> 'foo.com.au'"""
    if pd.isna(url):
        return np.nan
    d = re.sub(r"^[a-z]+://", "", str(url).strip().lower())
    d = re.sub(r"^www\d?\.", "", d)
    return d.split("/")[0].split("?")[0].split(":")[0] or np.nan

def tld_of(domain):
    return domain.rsplit(".", 1)[-1] if isinstance(domain, str) else np.nan

def clean_name(name):
    return re.sub(r"\s+", " ", str(name).replace("&amp;", "&")).strip(" ​")

LEGAL_SUFFIX = r"\b(pty|ltd|limited|llc|inc|incorporated|corp|corporation|co|company|plc|group|holdings|the)\b"
def norm_name(name):
    """Name key for tie-breaks and the review list (never used as a primary match)."""
    if pd.isna(name):
        return np.nan
    n = re.sub(LEGAL_SUFFIX, " ", clean_name(name).lower())
    return re.sub(r"[^a-z0-9]", "", n) or np.nan

def to_band(n):
    if pd.isna(n) or n <= 0:
        return np.nan
    return next(label for upper, label in BAND_EDGES if n <= upper)

def mystaff_region(country, state):
    if pd.isna(country):
        return np.nan
    c = COUNTRY_CODE.get(str(country).strip().lower(), "Other")
    if c == "AU":
        s = AU_STATE.get(str(state).strip().lower())
        return f"AU-{s}" if s else "AU"
    return c

def _name_evidence(name):
    n = clean_name(name).lower()
    for pat, country in NAME_COUNTRY:
        if re.search(pat, n):
            return country
    return "non-US" if re.search(NAME_NON_US, n) else None

def _report_evidence(text):
    t = str(text).lower() if pd.notna(text) else ""
    found = [c for c, pat in REPORT_COUNTRY.items() if re.search(pat, t)]
    if len(found) == 1 and not re.search(REPORT_US, t):
        return found[0]
    return "non-US" if found else None

def hubspot_region(domain, name, report_text):
    """Returns (region, region_source) following the evidence hierarchy in 0b."""
    t = tld_of(domain)
    if t in TLD_COUNTRY:
        return TLD_COUNTRY[t], "website TLD"
    by_name, by_report = _name_evidence(name), _report_evidence(report_text)
    if by_name not in (None, "non-US"):
        return by_name, "company name"
    if by_report not in (None, "non-US"):
        return by_report, "financial reports text"
    if "non-US" in (by_name, by_report):
        return np.nan, "non-US evidence, country unclear"
    if t in ASSUMED_TLD:
        return ASSUMED_TLD[t], f"assumed: .{t} → {ASSUMED_TLD[t]}"
    return np.nan, None

def profile(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "pct_filled": (df.notna().mean() * 100).round(1),
        "n_unique": df.nunique(),
        "example": df.apply(lambda s: s.dropna().iloc[0] if s.notna().any() else None),
    })

## 1. Extract
Pull raw records and rename them to the contract. In production, add the `db` / `api` branch here.
Nothing else in the notebook needs to change.

In [12]:
def extract_hubspot(cfg=SOURCES["hubspot"]):
    if SOURCE_MODE == "excel":
        raw = pd.read_excel(cfg["path"], sheet_name=cfg["sheet"], usecols=list(HUBSPOT_CONTRACT))
    else:
        raise NotImplementedError("HubSpot DB/API extract: TODO (companies endpoint, same fields as HUBSPOT_CONTRACT)")
    return validate_contract(raw.rename(columns=HUBSPOT_CONTRACT), HUBSPOT_CONTRACT, "hubspot")

def extract_mystaff(cfg=SOURCES["mystaff"]):
    if SOURCE_MODE == "excel":
        raw = pd.read_excel(cfg["path"], sheet_name=cfg["sheet"], usecols=list(MYSTAFF_CONTRACT))
        n = len(raw)
        raw = raw.dropna(subset=["Shortcode"])        # the export has ~1M formatted-but-empty rows
        print(f"mystaff: dropped {n - len(raw):,} empty spreadsheet rows")
    else:
        raise NotImplementedError("MyStaff DB extract: TODO (client table, same fields as MYSTAFF_CONTRACT)")
    return validate_contract(raw.rename(columns=MYSTAFF_CONTRACT), MYSTAFF_CONTRACT, "mystaff")

In [13]:
hs_raw = extract_hubspot()
ms_raw = extract_mystaff()          # ~1 min: large sheet
print("hubspot:", hs_raw.shape, "| mystaff:", ms_raw.shape)

mystaff: dropped 1,045,621 empty spreadsheet rows
hubspot: (484, 7) | mystaff: (1112, 16)


In [14]:
# INSPECT
display(hs_raw.head(5))
profile(hs_raw)

,hs_record_id,company_name,website_url,employee_count,industry_raw,annual_revenue,financial_reports_text
0,57889913861,1 to 1 Technology Ltd,1to1tech.co.uk,50,Management Consulting,10000000,Accounts made up to 2024?04?30 (filed 2024?09?10): PDF (...
1,57840517929,12768640 CANADA INC.,ash.agency,10,Online Media,1000000,Not found Not found
2,57841917290,263 ITS Operations Pty Ltd,mumbacloud.com,50,Computer Software,10000000,Not found. (Mumba Cloud is a private company. No public ...
3,4177537874,389 Services Pty Ltd,grangebp.com.au,19,Accounting,1000000,"Not found. (As a private company, 389 Services Pty Ltd d..."
4,9181185458,4Cabling Pty Ltd,4cabling.com.au,50,Computer Networking,10000000,"Not found. (As a private company, 4Cabling’s detailed fi..."


,dtype,pct_filled,n_unique,example
hs_record_id,int64,100.0,484,57889913861
company_name,str,100.0,484,1 to 1 Technology Ltd
website_url,str,100.0,484,1to1tech.co.uk
employee_count,int64,100.0,34,50
industry_raw,str,100.0,84,Management Consulting
annual_revenue,int64,100.0,10,10000000
financial_reports_text,str,100.0,425,Accounts made up to 2024?04?30 (filed 2024?09?10): PDF (...


In [15]:
# INSPECT
display(ms_raw.head(5))
profile(ms_raw)

,shortcode,company_name,active,crm_company_id,website_url,office_state,office_country,industry_hubspot,industry_rollup,cs_staff_current,seats_added_mtd,seats_added_m1,seats_added_m2,seats_added_m3,seats_added_m4,seats_added_m5
0,AUTHELANEHOUSE,The Lane House Melbourne Pty Ltd,Yes,6.028997e+10,https://thelanehouse.com.au,Victoria,AUSTRALIA,-,-,1.0,1.0,0.0,0.0,0.0,0.0,0.0
1,USLEVELTECH,"Level Technologies, LLC",Yes,5.848498e+10,NaN,Arkansas,UNITED STATES OF AMERICA,-,-,1.0,1.0,0.0,0.0,0.0,0.0,0.0
2,AUTESTTRACE,TestTrace Corp,Yes,5.971516e+10,https://testtrace.test,QLD,Australia,Facilities Services,Business Operations & Enterprise Solutions,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,AUTESTTRACKSERV,TestTrack Services,Yes,5.971605e+10,https://testtrackservices.test,QLD,Australia,Telecommunications,Telecommunications & Information Technology,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,USJLMARSHALL,"J.L. Marshall and Sons, Inc.",Yes,5.725089e+10,https://jlmarshall.com/,MA,United States Of America,Construction,Professional Advisory,2.0,0.0,2.0,0.0,0.0,0.0,0.0


,dtype,pct_filled,n_unique,example
shortcode,str,100.0,1112,AUTHELANEHOUSE
company_name,str,100.0,1112,The Lane House Melbourne Pty Ltd
active,str,100.0,1,Yes
crm_company_id,float64,100.0,1111,60289972358.0
website_url,object,99.8,1081,https://thelanehouse.com.au
office_state,object,99.7,140,Victoria
office_country,object,100.0,33,AUSTRALIA
industry_hubspot,str,100.0,97,-
industry_rollup,str,100.0,65,-
cs_staff_current,float64,100.0,55,1.0


## 2. Standardize
Each source → the shared **staging schema** (`STAGING_COLS`). All per-source cleaning and mapping happens here:
names, domains, industry, employee band, region. Rows that are filtered out are returned as `rejects` with a reason.

In [16]:
def standardize_hubspot(raw):
    df = raw.copy()
    rec_id = pd.to_numeric(df["hs_record_id"], errors="coerce").astype("Int64")
    domain = df["website_url"].map(to_domain)
    emp = pd.to_numeric(df["employee_count"], errors="coerce")
    emp = emp.where(emp > 0)                                   # 0 = unknown in HubSpot
    industry = df["industry_raw"].map(INDUSTRY_MAP)
    region = [hubspot_region(d, n, t) for d, n, t in zip(domain, df["company_name"], df["financial_reports_text"])]

    stg = pd.DataFrame({
        "source":            "HubSpot",
        "shortcode":         None,
        "hubspot_record_id": rec_id.astype(str),
        "crm_company_id":    rec_id,
        "company_name":      df["company_name"].map(clean_name),
        "name_key":          df["company_name"].map(norm_name),
        "website_domain":    domain,
        "industry_raw":      df["industry_raw"],
        "industry":          industry,
        "industry_source":   np.where(industry.notna(), "HubSpot Industry", None),
        "employee_count":    emp,
        "employee_band":     emp.map(to_band),
        "employee_source":   np.where(emp.notna(), "HubSpot", None),
        "region":            [r for r, _ in region],
        "region_source":     [s for _, s in region],
        "annual_revenue":    df["annual_revenue"],
        "cs_staff_current":  np.nan,
        "seats_added_6m":    np.nan,                         # prospects have no Cloudstaff seats
        "client_since":      pd.NaT,                         # prospects are not clients
        "client_tenure_band": None,
    }).reset_index(drop=True)
    return stg[STAGING_COLS]


def standardize_mystaff(raw):
    df = raw.copy()
    reason = pd.Series(None, index=df.index, dtype=object)
    reason[df["active"].astype(str).str.strip().str.lower().ne("yes")] = "inactive client"
    reason[df["company_name"].str.contains(TEST_ACCOUNT_PATTERN, case=False, na=False)] = "test / QA account"
    rejects = df[reason.notna()].assign(reject_reason=reason.dropna())
    df = df[reason.isna()]

    ind_h = df["industry_hubspot"].map(INDUSTRY_MAP)
    ind_r = df["industry_rollup"].where(~df["industry_rollup"].isin(PLACEHOLDER_ROLLUP)).map(ROLLUP_MAP)
    region = pd.Series([mystaff_region(c, s) for c, s in zip(df["office_country"], df["office_state"])], index=df.index)

    stg = pd.DataFrame({
        "source":            "MyStaff",
        "shortcode":         df["shortcode"].astype(str),
        "hubspot_record_id": None,
        "crm_company_id":    pd.to_numeric(df["crm_company_id"], errors="coerce").astype("Int64"),
        "company_name":      df["company_name"].map(clean_name),
        "name_key":          df["company_name"].map(norm_name),
        "website_domain":    df["website_url"].map(to_domain),
        "industry_raw":      df["industry_hubspot"].where(df["industry_hubspot"].ne("-"), df["industry_rollup"]),
        "industry":          ind_h.fillna(ind_r),
        "industry_source":   np.select([ind_h.notna(), ind_r.notna()],
                                       ["MyStaff Industry Hubspot", "MyStaff Industry Roll Up"], None),
        "employee_count":    np.nan,                         # MyStaff has no company headcount
        "employee_band":     np.nan,
        "employee_source":   None,
        "region":            region,
        "region_source":     np.where(region.notna(), "MyStaff office address", None),
        "annual_revenue":    np.nan,
        "cs_staff_current":  pd.to_numeric(df["cs_staff_current"], errors="coerce"),
        "seats_added_6m":    df[SEAT_WINDOW].apply(pd.to_numeric, errors="coerce").sum(axis=1, min_count=1),
        "client_since":      pd.to_datetime(df["client_since"], errors="coerce"),
        "client_tenure_band": df["client_tenure_band"].astype("string").str.replace(r"^\s*\d+\.\s*", "", regex=True).str.strip(),
    }).reset_index(drop=True)
    return stg[STAGING_COLS], rejects

In [17]:
stg_hs = standardize_hubspot(hs_raw)
stg_ms, ms_rejects = standardize_mystaff(ms_raw)
print("staging hubspot:", stg_hs.shape, "| staging mystaff:", stg_ms.shape, "| mystaff rejects:", len(ms_rejects))

staging hubspot: (484, 18) | staging mystaff: (1108, 18) | mystaff rejects: 4


In [18]:
# INSPECT: rejected MyStaff rows
ms_rejects[["shortcode", "company_name", "active", "industry_hubspot", "reject_reason"]]

,shortcode,company_name,active,industry_hubspot,reject_reason
2,AUTESTTRACE,TestTrace Corp,Yes,Facilities Services,test / QA account
3,AUTESTTRACKSERV,TestTrack Services,Yes,Telecommunications,test / QA account
6,AUCUSTOMERPORT2,Customer Portal - QA Test v5,Yes,Information Technology and Services,test / QA account
1102,RTGQA,ReadyToGo-QA,Yes,"Leisure, Travel & Tourism",test / QA account


In [19]:
# INSPECT: both sources now share one schema
display(stg_hs.head(5))
display(stg_ms.head(5))

,source,shortcode,hubspot_record_id,crm_company_id,company_name,name_key,website_domain,industry_raw,industry,industry_source,employee_count,employee_band,employee_source,region,region_source,annual_revenue,cs_staff_current,seats_added_6m
0,HubSpot,None,57889913861,57889913861,1 to 1 Technology Ltd,1to1technology,1to1tech.co.uk,Management Consulting,Professional Services,HubSpot Industry,50.0,11-50,HubSpot,UK,website TLD,10000000,NaN,NaN
1,HubSpot,None,57840517929,57840517929,12768640 CANADA INC.,12768640canada,ash.agency,Online Media,"Marketing, Media & Creative",HubSpot Industry,10.0,1-10,HubSpot,CA,company name,1000000,NaN,NaN
2,HubSpot,None,57841917290,57841917290,263 ITS Operations Pty Ltd,263itsoperations,mumbacloud.com,Computer Software,Technology,HubSpot Industry,50.0,11-50,HubSpot,AU,company name,10000000,NaN,NaN
3,HubSpot,None,4177537874,4177537874,389 Services Pty Ltd,389services,grangebp.com.au,Accounting,Accounting,HubSpot Industry,19.0,11-50,HubSpot,AU,website TLD,1000000,NaN,NaN
4,HubSpot,None,9181185458,9181185458,4Cabling Pty Ltd,4cabling,4cabling.com.au,Computer Networking,Technology,HubSpot Industry,50.0,11-50,HubSpot,AU,website TLD,10000000,NaN,NaN


,source,shortcode,hubspot_record_id,crm_company_id,company_name,name_key,website_domain,industry_raw,industry,industry_source,employee_count,employee_band,employee_source,region,region_source,annual_revenue,cs_staff_current,seats_added_6m
0,MyStaff,AUTHELANEHOUSE,None,60289972358,The Lane House Melbourne Pty Ltd,lanehousemelbourne,thelanehouse.com.au,-,NaN,NaN,NaN,NaN,None,AU-VIC,MyStaff office address,NaN,1.0,1.0
1,MyStaff,USLEVELTECH,None,58484978505,"Level Technologies, LLC",leveltechnologies,NaN,-,NaN,NaN,NaN,NaN,None,US,MyStaff office address,NaN,1.0,1.0
2,MyStaff,USJLMARSHALL,None,57250890407,"J.L. Marshall and Sons, Inc.",jlmarshallandsons,jlmarshall.com,Construction,Construction & Engineering,MyStaff Industry Hubspot,NaN,NaN,None,US,MyStaff office address,NaN,2.0,2.0
3,MyStaff,USUNLIMITEDHI,None,53016787343,"Unlimited Construction Services, Inc.",unlimitedconstructionservices,unlimitedhawaii.com,Information Technology and Services,Technology,MyStaff Industry Hubspot,NaN,NaN,None,US,MyStaff office address,NaN,1.0,1.0
4,MyStaff,AUAMPCONSW,None,9694287873,Australasian Medical Publishing Company Proprietary Limited,australasianmedicalpublishingproprietary,ampco.com.au,Information Technology and Services,Technology,MyStaff Industry Hubspot,NaN,NaN,None,AU-NSW,MyStaff office address,NaN,4.0,5.0


In [20]:
# INSPECT: raw industry values that didn't map to an industry group (new values -> add to INDUSTRY_GROUPS / ROLLUP_MAP)
unmapped = pd.concat([stg_hs, stg_ms])
unmapped = unmapped[unmapped["industry"].isna() & unmapped["industry_raw"].notna() & ~unmapped["industry_raw"].isin(PLACEHOLDER_ROLLUP | UNCLASSIFIED_INDUSTRY)]
unmapped.groupby(["industry_raw", "source"]).size().unstack(fill_value=0).assign(
    total=lambda d: d.sum(axis=1)).sort_values("total", ascending=False).head(25)

source,total
industry_raw,


In [21]:
# INSPECT: industry groups (with sector) by source, after the Roll Up fallback
ind = pd.concat([stg_hs, stg_ms])
ind = ind.assign(industry=ind["industry"].fillna("(blank)"))
t = ind.groupby(["industry", "source"]).size().unstack(fill_value=0).assign(total=lambda d: d.sum(axis=1))
t.insert(0, "sector", t.index.map(INDUSTRY_SECTOR))
display(t.sort_values("total", ascending=False))
ind.groupby(["source", "industry_source"], dropna=False).size().to_frame("rows")

source,sector,HubSpot,MyStaff,total
industry,,,,
Technology,Technology & Media,84,206,290
Real Estate,Property & Construction,61,167,228
Construction & Engineering,Property & Construction,62,82,144
Accounting,Financial & Professional Services,47,74,121
Financial Services,Financial & Professional Services,32,75,107
Retail & Consumer Goods,"Consumer, Hospitality & Services",42,61,103
"Marketing, Media & Creative",Technology & Media,22,58,80
Manufacturing & Industrial,"Industrial, Logistics & Energy",15,55,70
(blank),NaN,0,69,69


rows
source  industry_source               
HubSpot HubSpot Industry           484
MyStaff MyStaff Industry Hubspot   799
        MyStaff Industry Roll Up   240
        NaN                         69

In [22]:
# INSPECT: how HubSpot regions were decided
stg_hs["region_source"].value_counts(dropna=False).to_frame()

,count
region_source,
website TLD,235
assumed: .com → US,125
company name,72
financial reports text,22
NaN,18
"non-US evidence, country unclear",12


In [23]:
# INSPECT: .com companies NOT assumed US: what pointed elsewhere?
is_com = stg_hs["website_domain"].map(tld_of).eq("com")
(stg_hs[is_com & ~stg_hs["region_source"].str.startswith("assumed", na=False)]
    .merge(hs_raw.assign(hubspot_record_id=lambda d: d["hs_record_id"].astype(str))[["hubspot_record_id", "financial_reports_text"]],
           on="hubspot_record_id")
    .assign(financial_reports_text=lambda d: d["financial_reports_text"].astype(str).str[:100])
    [["company_name", "website_domain", "region", "region_source", "financial_reports_text"]])

,company_name,website_domain,region,region_source,financial_reports_text
0,263 ITS Operations Pty Ltd,mumbacloud.com,AU,company name,Not found. (Mumba Cloud is a private company. No public ...
1,A.C.N. 633 889 433 Pty Ltd,mspmagic.com,AU,company name,"Not found. As a private company, MSPMagic did not publis..."
2,Ambi Labs Limited,ambiclimate.com,NaN,"non-US evidence, country unclear",Not found (private company; no public financial filings/...
3,Arrow County Supplies Limited,arrowcounty.com,NaN,"non-US evidence, country unclear",Arrow County Supplies is private (now a Bunzl subsidiary...
4,AssetFuture Pty Ltd,assetfuture.com,AU,company name,No public financial filings are available for this priva...
...,...,...,...,...,...
76,Trimline Group,trimlinegroup.com,UK,financial reports text,2025-06-12 – Companies House: Dormant Company Accounts (...
77,Venture Drake Pty Ltd,venturedrake.com,AU,company name,"Not found. As a privately held company, Venture Drake do..."
78,Zala Group Pty Ltd,zalahair.com,AU,company name,"As a private company, Zala Group does not publish financ..."
79,Zedra Group Services Limited,zedra.com,NaN,"non-US evidence, country unclear",Not found. (ZEDRA is a private company and does not publ...


In [24]:
# INSPECT: evidence for the .com -> US assumption. Test domain endings against MyStaff's real office country.
chk = pd.DataFrame({"tld": stg_ms["website_domain"].map(tld_of), "actual": stg_ms["region"].str[:2]})
chk["guess"] = chk["tld"].map({**TLD_COUNTRY, **ASSUMED_TLD})
chk = chk.dropna(subset=["guess"]).assign(correct=lambda d: d["guess"] == d["actual"])
chk.groupby("tld").agg(clients=("correct", "size"), pct_correct=("correct", "mean")).round(2).sort_values("clients", ascending=False)

,clients,pct_correct
tld,,
au,539,1.00
com,437,0.68
nz,13,1.00
uk,13,1.00
ca,4,0.50
us,4,1.00
ie,2,0.50


## 3. Match: which HubSpot companies are already MyStaff clients?
One row per HubSpot record. Match keys, in priority order:
1. **CRM ID**: MyStaff `CRM Company ID` = HubSpot `Record ID` (exact)
2. **Website domain**, when it's unique to one MyStaff client
3. Shared domain (franchises / trusts) → tie-break on normalised name; if still unclear → `ambiguous` (not merged)

Company name is **not** a match key. Name-only hits are only recorded in `name_match_review`.

In [25]:
def shared_crm_ids(stg_ms):
    """CRM IDs carried by more than one MyStaff client: can't identify a single client."""
    ids = stg_ms["crm_company_id"]
    return set(ids[ids.duplicated(keep=False) & ids.notna()])

def match_sources(stg_hs, stg_ms):
    bad_ids = shared_crm_ids(stg_ms)
    by_id = stg_ms[stg_ms["crm_company_id"].notna() & ~stg_ms["crm_company_id"].isin(bad_ids)] \
                .set_index("crm_company_id")["shortcode"]
    by_domain = stg_ms.dropna(subset=["website_domain"]).groupby("website_domain")["shortcode"].apply(list)
    ms_idx = stg_ms.set_index("shortcode")
    client_by_name = stg_ms.dropna(subset=["name_key"]).drop_duplicates("name_key").set_index("name_key")["company_name"]

    rows = []
    for r in stg_hs.itertuples(index=False):
        hit = by_id.get(r.crm_company_id) if pd.notna(r.crm_company_id) else None
        cands = by_domain.get(r.website_domain, []) if isinstance(r.website_domain, str) else []
        if hit is not None:
            code_, mtype = hit, ("CRM ID + domain" if hit in cands else "CRM ID")
        elif len(cands) == 1:
            code_, mtype = cands[0], "Website domain"
        elif len(cands) > 1:
            same = [c for c in cands if ms_idx.at[c, "name_key"] == r.name_key]
            code_, mtype = (same[0], "Website domain (name tie-break)") if len(same) == 1 else (None, AMBIGUOUS)
        else:
            code_, mtype = None, None
        rows.append({
            "hubspot_record_id": r.hubspot_record_id,
            "hs_company_name":   r.company_name,
            "website_domain":    r.website_domain,
            "shortcode":         code_,
            "ms_client":         ms_idx.at[code_, "company_name"] if code_ else None,
            "match_type":        mtype,
            "candidate_clients": " | ".join(ms_idx.loc[cands, "company_name"]) if mtype == AMBIGUOUS else None,
            "name_match_review": client_by_name.get(r.name_key) if mtype is None and pd.notna(r.name_key) else None,
        })
    return pd.DataFrame(rows)

In [26]:
matches = match_sources(stg_hs, stg_ms)
matches["match_type"].value_counts(dropna=False).to_frame("hubspot_rows")

,hubspot_rows
match_type,
NaN,217
CRM ID + domain,200
Website domain,45
CRM ID,17
Website domain (name tie-break),3
Website domain (ambiguous),2


In [27]:
# INSPECT: CRM IDs excluded from matching because several MyStaff clients share them
stg_ms[stg_ms["crm_company_id"].isin(shared_crm_ids(stg_ms))][["shortcode", "company_name", "crm_company_id", "website_domain"]]

,shortcode,company_name,crm_company_id,website_domain
266,RAMGMT,RA Management NSW Pty Ltd,44331567279,roberts.co
291,TIASCHOOL,TIA School of Allied Health,44331567279,tiamedical.com


In [28]:
# INSPECT: matched pairs. Domain-only matches first, since those are the ones to sanity-check.
(matches.dropna(subset=["shortcode"])
        .sort_values("match_type", key=lambda s: s.map(MATCH_RANK), ascending=False)
        [["match_type", "hs_company_name", "ms_client", "website_domain", "hubspot_record_id", "shortcode"]]
        .head(60))

,match_type,hs_company_name,ms_client,website_domain,hubspot_record_id,shortcode
433,Website domain (name tie-break),The Trustee for Resmark Unit Trust,The Trustee for Resmark Unit Trust,realmark.com.au,57842115096,REALMARK
323,Website domain (name tie-break),Place Head Office Administration Pty Ltd,Place Head Office Administration Pty Ltd,eplace.com.au,35841412693,PLACEHQ
434,Website domain (name tie-break),The Trustee for the Melcorp Strata Unit Trust,The Trustee for the Melcorp Strata Unit Trust,giulianogroup.com.au,57840433445,GIULIANOGROUP
456,Website domain,Veritas Investments Inc.,Veritas Investments Inc.,veritasinvestments.com,21856557451,VERITAS
459,Website domain,Vietell Pty Ltd,Vietell Pty Ltd,charlesstuart.com.au,4269295147,VIETELL
67,Website domain,BSC Global Inc,BSC Global Inc,bscglobal.co,57839776658,BSCGLOBAL
54,Website domain,Beverly Hills - Mazzhills Enterprises Pty Ltd / Bexley N...,Beverly Hills - Mazzhills Enterprises Pty Ltd / Bexley N...,prdbn.com.au,57840509365,BEXLEY
407,Website domain,TEC Group Inc.,TEC Group Inc.,tec.biz,57840854712,TECGROUP
457,Website domain,Vertical Internet Media Limited,Vertical Internet Media Limited,verticalinternetradio.com.au,57898266942,HPR
352,Website domain,Reitsma Constructions Pty Ltd,Reitsma Constructions Pty Ltd,reitsma.com,57842118574,REITSMA


In [29]:
# INSPECT: ambiguous (domain shared by several clients, no name match). Become Organic, NOT merged.
matches[matches["match_type"] == AMBIGUOUS][["hs_company_name", "website_domain", "candidate_clients"]]

,hs_company_name,website_domain,candidate_clients
83,Cattell Family Trust T/AS Exacta Enterprises Pty Ltd,prd.com.au,Eastern Suburbs Realty Pty Ltd | Plodders Pty Ltd dba PR...
88,Choice Mortgage Power Pty Ltd,ybr.com.au,Reidy Property Pty Ltd | The Trustee for YBR CARLTON TRUST


In [30]:
# INSPECT: review only. Same name as a client but no CRM ID / domain match. These stay New Logo.
matches[matches["name_match_review"].notna()][["hs_company_name", "name_match_review", "website_domain"]]

,hs_company_name,name_match_review,website_domain
16,All Risk Protection Pty Ltd,All Risk Protection Pty Ltd,oraclegroup.com.au
68,Business Automation Works Pty Ltd,Business Automation Works Pty Ltd,teambaw.com.au
90,City Chic Collective Limited,City Chic Collective Limited,citychic.com.au
102,Colab6 Pty Ltd,Colab6 Pty Ltd,colab6.com
158,Fusion5 Pty Limited,Fusion5 Pty Limited,integration.works
184,"Hilton Grand Vacations Club, LLC","Hilton Grand Vacations Club, LLC",hiltongrandvacations.com
188,Homewave Inc.,Homewave Inc.,212.ca
227,"KEC Miriz Inc., Ltd","KEC Miriz Inc., Ltd",chattyonline.com
254,McConnell Dowell Constructors (Aust) Pty Ltd,McConnell Dowell Constructors (Aust) Pty Ltd,mcdgroup.com
258,MedRecruit International LP,MedRecruit International LP,medenterprises.com


## 4. Merge (survivorship rules)
* **Matched** → one row, goal **Organic Growth**. The MyStaff record survives (shortcode, name, office region).
  HubSpot fills only what MyStaff lacks:

  | Field | Rule |
  |---|---|
  | `employee_band` | HubSpot (MyStaff has no company headcount) |
  | `industry` | MyStaff; HubSpot only if MyStaff is blank |
  | `region` | MyStaff office address; HubSpot only if MyStaff is blank |
  | `annual_revenue`, `hubspot_record_id`, `hs_company_name` | HubSpot |

  If several HubSpot records match one client, the best match (by `MATCH_RANK`) supplies values and all their IDs are kept.
* **Ambiguous** → HubSpot row kept as its own record, goal relabelled **Organic Growth**, with candidate clients in `note`.
* **Unmatched HubSpot** → **New Logo**.
* **Unmatched MyStaff** → **Organic Growth**.

In [31]:
def merge_sources(stg_hs, stg_ms, matches):
    m = (matches.dropna(subset=["shortcode"])
                .assign(_rank=lambda d: d["match_type"].map(MATCH_RANK))
                .sort_values(["shortcode", "_rank"]))
    best = m.drop_duplicates("shortcode").set_index("shortcode")
    all_ids = m.groupby("shortcode")["hubspot_record_id"].agg(", ".join)

    # ---- Organic: every MyStaff client, enriched where matched
    org = stg_ms.copy()
    org["goal"] = GOALS["mystaff"]
    org[["match_type", "hs_company_name", "name_match_review", "note"]] = None
    hs_for = (stg_hs.set_index("hubspot_record_id")
                    .reindex(org["shortcode"].map(best["hubspot_record_id"]))
                    .set_axis(org.index))
    is_m = org["shortcode"].isin(best.index)

    org.loc[is_m, "source"] = "MyStaff + HubSpot"
    org.loc[is_m, "match_type"] = org.loc[is_m, "shortcode"].map(best["match_type"])
    org.loc[is_m, "hubspot_record_id"] = org.loc[is_m, "shortcode"].map(all_ids)
    org.loc[is_m, "hs_company_name"] = hs_for.loc[is_m, "company_name"]
    org.loc[is_m, "annual_revenue"] = hs_for.loc[is_m, "annual_revenue"]

    f = is_m & hs_for["employee_count"].notna()                         # employees: HubSpot
    org.loc[f, "employee_count"] = hs_for.loc[f, "employee_count"]
    org.loc[f, "employee_source"] = "HubSpot (merged)"
    org["employee_band"] = org["employee_count"].map(to_band)

    f = is_m & org["industry"].isna() & hs_for["industry"].notna()      # industry: MyStaff, else HubSpot
    org.loc[f, "industry"] = hs_for.loc[f, "industry"]
    org.loc[f, "industry_raw"] = hs_for.loc[f, "industry_raw"]
    org.loc[f, "industry_source"] = "HubSpot Industry (merged)"

    f = is_m & org["region"].isna() & hs_for["region"].notna()          # region: MyStaff, else HubSpot
    org.loc[f, "region"] = hs_for.loc[f, "region"]
    org.loc[f, "region_source"] = hs_for.loc[f, "region_source"] + " (merged)"

    # ---- HubSpot rows that were not merged
    mt = matches.set_index("hubspot_record_id")
    hs = stg_hs[~stg_hs["hubspot_record_id"].isin(m["hubspot_record_id"])].copy()
    is_amb = hs["hubspot_record_id"].map(mt["match_type"]).eq(AMBIGUOUS)
    hs["goal"] = np.where(is_amb, GOALS["mystaff"], GOALS["hubspot"])
    hs["match_type"] = hs["hubspot_record_id"].map(mt["match_type"])
    hs["hs_company_name"] = None
    hs["name_match_review"] = hs["hubspot_record_id"].map(mt["name_match_review"])
    hs["note"] = np.where(is_amb, "Domain shared by several MyStaff clients; not merged. Candidates: "
                          + hs["hubspot_record_id"].map(mt["candidate_clients"]).fillna(""), None)

    recon = pd.Series({
        "HubSpot records in":            len(stg_hs),
        "  merged into a MyStaff client": len(m),
        "  ambiguous → Organic (own row)": int(is_amb.sum()),
        "  New Logo":                     int((~is_amb).sum()),
        "MyStaff clients in":            len(stg_ms),
        "  of which enriched by HubSpot": int(is_m.sum()),
    })
    assert recon.iloc[0] == recon.iloc[1] + recon.iloc[2] + recon.iloc[3], "HubSpot rows don't reconcile"
    return pd.concat([hs, org], ignore_index=True), recon

In [32]:
merged, recon = merge_sources(stg_hs, stg_ms, matches)
recon.to_frame("rows")

,rows
HubSpot records in,484
merged into a MyStaff client,265
ambiguous → Organic (own row),2
New Logo,217
MyStaff clients in,1108
of which enriched by HubSpot,265


In [33]:
# INSPECT: merged rows: MyStaff identity + HubSpot fill-ins
merged[merged["source"] == "MyStaff + HubSpot"][
    ["shortcode", "company_name", "hs_company_name", "match_type", "industry", "industry_source",
     "employee_band", "employee_source", "region", "hubspot_record_id"]].head(25)

,shortcode,company_name,hs_company_name,match_type,industry,industry_source,employee_band,employee_source,region,hubspot_record_id
580,VISORY,Visory Pty Ltd,Visory Pty Ltd,CRM ID + domain,"Marketing, Media & Creative",MyStaff Industry Hubspot,11-50,HubSpot (merged),AU-VIC,9525532484
616,GURNERGRP,GURNER RE SALES PTY LTD,GURNER RE SALES PTY LTD,CRM ID + domain,"Marketing, Media & Creative",MyStaff Industry Hubspot,201-500,HubSpot (merged),AU-VIC,4178095013
620,BOLDICT,Bold ICT Pty Ltd,Bold ICT Pty Ltd,CRM ID + domain,Real Estate,MyStaff Industry Roll Up,1-10,HubSpot (merged),AU-VIC,21897218331
626,REINSWAU,THE REAL ESTATE INSTITUTE OF NEW SOUTH WALES LIMITED,The Real Estate Institute Of New South Wales Limited Doi...,Website domain,Real Estate,MyStaff Industry Hubspot,11-50,HubSpot (merged),AU-NSW,21501176445
631,FAIRCAPITAL,Fair Capital LLC,Fair Capital LLC,CRM ID + domain,Technology,MyStaff Industry Hubspot,11-50,HubSpot (merged),US,8534447289
642,TECGROUP,TEC Group Inc.,TEC Group Inc.,Website domain,Financial Services,MyStaff Industry Hubspot,500+,HubSpot (merged),US,57840854712
654,IBISWORLD,IBISWorld Pty Ltd,IBISWorld Pty Ltd,CRM ID + domain,Professional Services,HubSpot Industry (merged),201-500,HubSpot (merged),AU-VIC,10215476296
661,MELOTTIMEDIA,Melotti Media Pty Ltd,Melotti Media Pty Ltd,CRM ID + domain,Retail & Consumer Goods,MyStaff Industry Hubspot,1-10,HubSpot (merged),AU-NSW,4177538468
662,TORTICITY,"Torticity, LLC","Torticity, LLC",CRM ID + domain,Technology,MyStaff Industry Hubspot,201-500,HubSpot (merged),US,31015985076
674,ALLAMERICAN,All American generator Services,All American generator Services,CRM ID + domain,Accounting,MyStaff Industry Hubspot,1-10,HubSpot (merged),US,31112290325


## 5. Conform to the output schema
Builds `account_id` (`MS-<shortcode>` for clients, `HS-<record id>` for HubSpot-only), adds the enrichment and scoring
columns as **empty placeholders**, and orders the columns by group: company → enrichment → scoring → lineage.

In [34]:
def conform_to_output(merged):
    out = merged.copy()
    out["account_id"] = np.where(out["shortcode"].notna(),
                                 "MS-" + out["shortcode"].astype(str),
                                 "HS-" + out["hubspot_record_id"].astype(str))
    out["industry_sector"] = out["industry"].map(INDUSTRY_SECTOR)
    for c in ENRICHMENT_COLS + SCORING_COLS + PROPENSITY_COLS + ["enrichment_source"]:
        if c not in out:                       # placeholders; real signals (seats_added_6m) are kept
            out[c] = np.nan
    return out[OUTPUT_COLS].sort_values(["goal", "company_name"]).reset_index(drop=True)

def cols_in(*groups):
    """Select columns by group, e.g. output[cols_in('1_company', '4_lineage')]"""
    return [c for c in OUTPUT_COLS if COLUMN_GROUPS[c] in groups]

In [35]:
output = conform_to_output(merged)
print(output.shape)
pd.crosstab(output["source"], output["goal"], margins=True)

(1327, 56)


goal,New Logo,Organic Growth,All
source,,,
HubSpot,217,2,219
MyStaff,0,843,843
MyStaff + HubSpot,0,265,265
All,217,1110,1327


In [36]:
# INSPECT: what this pipeline populates (company + lineage)
output[cols_in("1_company", "4_lineage")].head(20)

,account_id,company_name,goal,industry,employee_band,region,source,shortcode,hubspot_record_id,crm_company_id,match_type,hs_company_name,website_domain,industry_sector,industry_raw,industry_source,employee_count,employee_source,region_source,annual_revenue,cs_staff_current,name_match_review,note,enrichment_source
0,HS-4177537874,389 Services Pty Ltd,New Logo,Accounting,11-50,AU,HubSpot,None,4177537874,4177537874,NaN,None,grangebp.com.au,Financial & Professional Services,Accounting,HubSpot Industry,19.0,HubSpot,website TLD,1000000.0,NaN,NaN,NaN,NaN
1,HS-9181185458,4Cabling Pty Ltd,New Logo,Technology,11-50,AU,HubSpot,None,9181185458,9181185458,NaN,None,4cabling.com.au,Technology & Media,Computer Networking,HubSpot Industry,50.0,HubSpot,website TLD,10000000.0,NaN,NaN,NaN,NaN
2,HS-16740821091,A.C.N. 633 889 433 Pty Ltd,New Logo,Technology,1-10,AU,HubSpot,None,16740821091,16740821091,NaN,None,mspmagic.com,Technology & Media,Information Technology and Services,HubSpot Industry,10.0,HubSpot,company name,1000000.0,NaN,NaN,NaN,NaN
3,HS-57844246813,ADESA SERVICES PTY LTD,New Logo,Construction & Engineering,11-50,AU,HubSpot,None,57844246813,57844246813,NaN,None,adesaservices.com.au,Property & Construction,Construction,HubSpot Industry,50.0,HubSpot,website TLD,10000000.0,NaN,NaN,NaN,NaN
4,HS-4319585420,AMR Motors Pty Ltd,New Logo,Manufacturing & Industrial,11-50,AU,HubSpot,None,4319585420,4319585420,NaN,None,amrmotors.com.au,"Industrial, Logistics & Energy",Motor Vehicle Manufacturing,HubSpot Industry,50.0,HubSpot,website TLD,10000000.0,NaN,NaN,NaN,NaN
5,HS-8337617848,AR Resources Inc.,New Logo,Business & Consumer Services,11-50,US,HubSpot,None,8337617848,8337617848,NaN,None,arresourcesinc.com,"Consumer, Hospitality & Services",Consumer Services,HubSpot Industry,50.0,HubSpot,assumed: .com → US,10000000.0,NaN,NaN,NaN,NaN
6,HS-4675979868,ARG LLC,New Logo,Transport & Logistics,1-10,NaN,HubSpot,None,4675979868,4675979868,NaN,None,argglobal.net,"Industrial, Logistics & Energy",Logistics and Supply Chain,HubSpot Industry,10.0,HubSpot,NaN,10000000.0,NaN,NaN,NaN,NaN
7,HS-6947188532,"ATKG Advisors, LLC",New Logo,Accounting,51-200,US,HubSpot,None,6947188532,6947188532,NaN,None,atkgcpa.com,Financial & Professional Services,Accounting,HubSpot Industry,100.0,HubSpot,assumed: .com → US,50000000.0,NaN,NaN,NaN,NaN
8,HS-57844247725,Accora,New Logo,Healthcare & Wellness,201-500,UK,HubSpot,None,57844247725,57844247725,NaN,None,accora.care,Health & Education,Hospital & Health Care,HubSpot Industry,250.0,HubSpot,financial reports text,50000000.0,NaN,NaN,NaN,NaN
9,HS-57839701870,Adkwik,New Logo,Manufacturing & Industrial,1-10,UK,HubSpot,None,57839701870,57839701870,NaN,None,adkwik.co.uk,"Industrial, Logistics & Energy",Chemicals,HubSpot Industry,10.0,HubSpot,website TLD,10000000.0,NaN,NaN,NaN,NaN


In [37]:
# INSPECT: what signals / scoring still need to populate (empty, except seats_added_6m from MyStaff)
output[cols_in("2_enrichment", "3_scoring")].head(5)

,web_visits_30d,team_builder_visit,form_fill_30d,news_signal,news_signal_date,job_postings_30d,hiring_spike,diamond_referral,prior_bpo_flag,industry_trend,seats_added_6m,industry_modifier,web_signal (0-1),news_jobs_signal (0-1),web_pts,news_jobs_pts,diamond_pts,bpo_pts,base_points,priority_score,estimated_fields,data_coverage,hot_trigger,tier,assigned_team,rank_in_team_list,sort_key,seat_signal (0-1),seat_pts,score_before_bonus,act_now_bonus,act_now_triggers
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 6. Data quality checks
Any failed `error` check stops the pipeline. `warn` and `info` checks are reported but don't stop it.

In [38]:
def run_dq_checks(out, stg_hs, stg_ms):
    rows = []
    def check(name, level, ok, detail=""):
        rows.append({"check": name, "level": level, "passed": bool(ok), "detail": detail})

    check("account_id unique", "error", out["account_id"].is_unique,
          f"{out['account_id'].duplicated().sum()} duplicates")

    hs_ids_out = set(out.loc[out["source"] == "HubSpot", "hubspot_record_id"]) | {
        i for s in out.loc[out["source"] == "MyStaff + HubSpot", "hubspot_record_id"] for i in s.split(", ")}
    check("every HubSpot record accounted for", "error", hs_ids_out == set(stg_hs["hubspot_record_id"]),
          f"{len(hs_ids_out)} of {len(stg_hs)}")
    check("every active MyStaff client present", "error", set(out["shortcode"].dropna()) == set(stg_ms["shortcode"]),
          f"{out['shortcode'].notna().sum()} of {len(stg_ms)}")

    known = shared_crm_ids(stg_ms)
    ids = out["crm_company_id"].dropna()
    dup = ids[ids.duplicated() & ~ids.isin(known)]
    check("no company duplicated via CRM ID", "error", dup.empty,
          f"{len(dup)} duplicates (ignoring {len(known)} CRM ID(s) already shared inside MyStaff)")

    check("goal values valid", "error", out["goal"].isin(GOALS.values()).all())
    check("industry is a known industry group or blank", "error", out["industry"].dropna().isin(INDUSTRY_GROUPS).all())
    check("industry_sector set wherever industry is", "error", (out["industry"].isna() == out["industry_sector"].isna()).all())
    unknown_raw = out.loc[out["industry"].isna() & out["industry_raw"].notna()
                          & ~out["industry_raw"].isin(UNCLASSIFIED_INDUSTRY | PLACEHOLDER_ROLLUP), "industry_raw"]
    check("no unrecognised raw industry values", "warn", unknown_raw.empty, f"{sorted(unknown_raw.unique())[:10]}")
    check("employee_band in mock vocabulary or blank", "error", out["employee_band"].dropna().isin(REF_BANDS).all())
    extra = sorted(set(out["region"].dropna()) - set(REF_REGIONS))
    check("region in mock vocabulary or blank", "warn", not extra, f"extra values: {extra}")
    check("enrichment + scoring columns empty", "info", out[[c for c in ENRICHMENT_COLS + SCORING_COLS if c not in REAL_SIGNAL_COLS]].isna().all().all(),
          "populated by the enrichment stage")
    return pd.DataFrame(rows)

def assert_dq(dq):
    failed = dq[(dq["level"] == "error") & ~dq["passed"]]
    if not failed.empty:
        raise AssertionError(f"DQ errors:\n{failed.to_string()}")

In [39]:
dq = run_dq_checks(output, stg_hs, stg_ms)
display(dq)
assert_dq(dq)

,check,level,passed,detail
0,account_id unique,error,True,0 duplicates
1,every HubSpot record accounted for,error,True,484 of 484
2,every active MyStaff client present,error,True,1108 of 1108
3,no company duplicated via CRM ID,error,True,0 duplicates (ignoring 1 CRM ID(s) already shared inside...
4,goal values valid,error,True,
5,industry is a known industry group or blank,error,True,
6,industry_sector set wherever industry is,error,True,
7,no unrecognised raw industry values,warn,True,[]
8,employee_band in mock vocabulary or blank,error,True,
9,region in mock vocabulary or blank,warn,False,"extra values: ['AE', 'AU', 'AU-ACT', 'AU-NT', 'AU-TAS', ..."


In [40]:
# INSPECT: share of blank values per company field, by goal
fields = ["industry", "employee_band", "region"]
output.groupby("goal")[fields].apply(lambda d: d.isna().mean().round(3))

,industry,employee_band,region
goal,,,
New Logo,0.000,0.083,0.046
Organic Growth,0.031,0.788,0.000


In [41]:
# INSPECT: value distributions by goal
for c in fields:
    print(f"\n--- {c} ---")
    print(output.groupby("goal")[c].value_counts(dropna=False).unstack(0).fillna(0).astype(int).to_string())


--- industry ---
goal                           New Logo  Organic Growth
industry                                               
Accounting                           18              82
Business & Consumer Services          6              20
Construction & Engineering           38              83
Education & Training                  5              32
Energy & Environment                  5              16
Financial Services                   15              76
Healthcare & Wellness                14              41
Hospitality, Travel & Leisure         8              28
Insurance                             2              21
Legal                                 6              24
Manufacturing & Industrial           10              55
Marketing, Media & Creative           4              61
Non-profit & Public Sector            1               6
Professional Services                 2              64
Real Estate                          21             173
Retail & Consumer Goods       

## 7. Signals (`add_signals`)
Fills group **2_enrichment**.

| Column(s) | Source today | Production source (to confirm) | Join key |
|---|---|---|---|
| `seats_added_6m` | **Real**: MyStaff staff added, month to date + 5 previous months (stage 1–2) | MyStaff DB | `shortcode` |
| `web_visits_30d`, `team_builder_visit`, `form_fill_30d` | **MOCK** | Website analytics / HubSpot activity | `website_domain`, `crm_company_id` |
| `news_signal`, `news_signal_date` | **MOCK** | News scraper (`News Scraper/` at the repo root, already outputs both fields) | `company_name`, `website_domain` |
| `job_postings_30d`, `hiring_spike` | **MOCK** | Job postings source (TBD) | `company_name`, `website_domain` |
| `diamond_referral`, `prior_bpo_flag` (= Competitor client) | **MOCK** | TBD (New Logo only) | `crm_company_id` |
| `industry_trend` | Mock values (`INDUSTRY_TREND`) | Industry news | `industry` |

The mock generator uses the mock workbook's rules: a latent "intent" per account, the same unknown rates by goal, and
Diamond / Competitor = `N/A` for Organic Growth. The random seed is set per `account_id`, so every account gets the same
values on every run. **Mock values are fictional.** Every row is tagged in `enrichment_source`.

In [42]:
MOCK_SIGNAL_COLS = ["web_visits_30d", "team_builder_visit", "form_fill_30d", "news_signal", "news_signal_date",
                    "job_postings_30d", "hiring_spike", "diamond_referral", "prior_bpo_flag"]

# How common the mock signals are. The workbook generator's own rates gave about a third of accounts a trigger event
# (Diamond referral 16% of prospects, Team Builder visit / heavy visits 15% of clients). Tuned 2026-11-13 so about 10% of
# each list has a trigger (ACT_NOW_TARGET_SHARE), in plausible proportions (simulated, 40,000 draws per goal; any trigger
# 9.4% New Logo, 8.6% Organic Growth):
#   New Logo: Diamond referral 2.9%, funding / PE 2.0%, hiring spike 1.7%, restructuring / leadership 1.3%,
#             Team Builder visit 0.8%, heavy visits 0.7%, form fill 0.6%
#   Organic Growth: funding / PE 2.9%, Team Builder visit 1.8%, restructuring / leadership 1.7%, heavy visits 1.5%, form fill 1.2%
# Workbook rates, for reference: visits_scale 26, tb_rate 0.45, ff_rate 0.28, jobs_scale 26, diamond_base 0.02, diamond_rate 0.25.
MOCK_RATES = {
    "visits_scale": 16,       # mean visits in 30 days = intent x this (20+ visits = heavy-visits trigger)
    "tb_rate": 0.055,         # chance of a Team Builder visit = intent x this (for accounts with visits)
    "ff_rate": 0.04,          # chance of a form fill = intent x this (for accounts with visits)
    "jobs_scale": 15,         # mean job postings in 30 days = intent x this (18+ = hiring trigger on New Logo)
    "diamond_base": 0.005,    # chance of a Diamond referral = diamond_base + intent x diamond_rate (New Logo)
    "diamond_rate": 0.045,
}

def _mock_signals(goal, rng, as_of):
    """One account's raw signals, using the mock generator's rules (generate_mock_workbook.py)."""
    newlogo = goal == GOALS["hubspot"]
    intent = rng.betavariate(1.05, 1.0)                      # latent buying / growth intent, 0-1
    if rng.random() < (0.68 if newlogo else 0.28):           # website data unknown
        visits = tb = ff = None
    else:
        visits = int(max(0, rng.gauss(intent * MOCK_RATES["visits_scale"], 4))) if rng.random() < 0.35 + intent else 0
        tb = "Yes" if (visits > 0 and rng.random() < intent * MOCK_RATES["tb_rate"]) else "No"
        ff = int(rng.random() < intent * MOCK_RATES["ff_rate"]) * rng.randint(1, 3) if visits > 0 else 0
    if rng.random() < (0.55 if newlogo else 0.38):           # news / jobs data unknown
        news = ndate = jobs = spike = None
    else:
        if rng.random() < 0.15 + intent * 0.8:
            cats = [c for c in NEWS_STRENGTH if c != "None"]
            news = rng.choices(cats, weights=[intent, intent, intent * 0.8, 0.6, 0.5, 0.4, 0.3])[0]
            ndate = as_of - pd.Timedelta(days=int(rng.uniform(3, 60 + (1 - intent) * 280)))
        else:
            news, ndate = "None", None                       # checked, nothing found
        jobs = None if rng.random() < 0.30 else int(max(0, rng.gauss(intent * MOCK_RATES["jobs_scale"], 5)))
        spike = None if jobs is None else ("Yes" if jobs >= 18 and rng.random() < 0.7 else "No")
    if newlogo:
        r1 = rng.random()
        dia = "Yes" if r1 < MOCK_RATES["diamond_base"] + intent * MOCK_RATES["diamond_rate"] else ("No" if r1 < 0.45 else None)
        r2 = rng.random()
        bpo = "Yes" if r2 < 0.05 + intent * 0.5 else ("No" if r2 < 0.40 else None)
    else:
        dia = bpo = "N/A"                                    # not applicable to Organic Growth
    return dict(zip(MOCK_SIGNAL_COLS, [visits, tb, ff, news, ndate, jobs, spike, dia, bpo]))

def generate_mock_enrichment(out, seed=MOCK_SEED, as_of=SCORING_AS_OF):
    """SYNTHETIC signals, seeded per account_id (same values every run, independent of row order)."""
    rows = [_mock_signals(goal, random.Random(f"{seed}:{acc}"), as_of)
            for acc, goal in zip(out["account_id"], out["goal"])]
    sig = pd.DataFrame(rows, index=out.index)
    for c in ["web_visits_30d", "form_fill_30d", "job_postings_30d"]:
        sig[c] = pd.to_numeric(sig[c]).astype("Int64")
    sig["news_signal_date"] = pd.to_datetime(sig["news_signal_date"])
    return sig[MOCK_SIGNAL_COLS]

def add_signals(out, mode=ENRICHMENT_MODE):
    """Stage 7: fill the 2_enrichment group. seats_added_6m is already real (MyStaff, stages 1-2)."""
    out = out.copy()
    if mode == "mock":
        sig = generate_mock_enrichment(out)
        for c in MOCK_SIGNAL_COLS:
            out[c] = sig[c]
        out["enrichment_source"] = f"MOCK signals (synthetic, seed={MOCK_SEED}); seats_added_6m real (MyStaff)"
    else:
        raise NotImplementedError("live enrichment sources not built yet")
    out["industry_trend"] = out["industry"].map(INDUSTRY_TREND)
    return out

In [43]:
output = add_signals(output)
output[["account_id", "goal"] + cols_in("2_enrichment")].head(10)

,account_id,goal,web_visits_30d,team_builder_visit,form_fill_30d,news_signal,news_signal_date,job_postings_30d,hiring_spike,diamond_referral,prior_bpo_flag,industry_trend,seats_added_6m
0,HS-4177537874,New Logo,<NA>,NaN,<NA>,Restructuring,2026-09-12,<NA>,NaN,No,NaN,0.7,NaN
1,HS-9181185458,New Logo,<NA>,NaN,<NA>,NaN,NaT,<NA>,NaN,Yes,NaN,0.8,NaN
2,HS-16740821091,New Logo,<NA>,NaN,<NA>,NaN,NaT,<NA>,NaN,NaN,NaN,0.8,NaN
3,HS-57844246813,New Logo,16,Yes,2,NaN,NaT,<NA>,NaN,NaN,NaN,0.6,NaN
4,HS-4319585420,New Logo,<NA>,NaN,<NA>,NaN,NaT,<NA>,NaN,NaN,Yes,NaN,NaN
5,HS-8337617848,New Logo,8,No,2,NaN,NaT,<NA>,NaN,Yes,NaN,NaN,NaN
6,HS-4675979868,New Logo,12,No,0,NaN,NaT,<NA>,NaN,No,NaN,0.7,NaN
7,HS-6947188532,New Logo,<NA>,NaN,<NA>,None,NaT,2,No,NaN,NaN,0.7,NaN
8,HS-57844247725,New Logo,<NA>,NaN,<NA>,NaN,NaT,<NA>,NaN,NaN,NaN,0.8,NaN
9,HS-57839701870,New Logo,13,No,0,PE buyout,2026-07-28,<NA>,NaN,No,NaN,NaN,NaN


In [44]:
# INSPECT: real seat growth (Organic Growth), in the dashboard's bands
og = output["goal"] == GOALS["mystaff"]
seat_band = pd.cut(output.loc[og, "seats_added_6m"], [-1, 0, 5, 10, np.inf], labels=["0", "1-5", "6-10", "11+"])
display(seat_band.value_counts(dropna=False).sort_index().to_frame("Organic Growth clients"))
output.loc[og, "seats_added_6m"].describe().to_frame("seats added (~6 months)").T

,Organic Growth clients
seats_added_6m,
0,751
1-5,320
6-10,19
11+,18
NaN,2


,count,mean,std,min,25%,50%,75%,max
seats added (~6 months),1108.0,1.052347,4.821939,0.0,0.0,0.0,1.0,119.0


## 7b. Propensity signals (`add_propensity`)
Fills group **5_propensity** with fit scores from the **Client Propensity** project (`Client Propensity/`, outside this
folder). They say how likely clients or prospects *like this one* are to grow. Their **bands** feed the **Fit** points of
`priority_score` (`dashboard_v3`, stage 8) and the fit chips on the page (stage 11); BUSINESS_RULES §11.2b.

| `PROPENSITY_MODE` | What happens |
|---|---|
| `"real"` | Reads the newest `*_Propensity_Extract_propensity_accounts.csv` and `*_prospect_profiles.csv` in `DATA_DIR` (Client Propensity's `run_pipeline.py` copies them here), plus `run_info` from its `.xlsx` |
| `"mock"` | Synthetic values, seeded per `account_id`. **Fictional.** Tagged in `propensity_source`, files get `_MOCK_SIGNALS` |
| `"off"` | Columns stay blank |

| Rows | Join | Columns filled |
|---|---|---|
| Organic Growth | `account_id` (= `MS-` + shortcode), one-to-one | `fit_seat_milestone*` (the client's next seat milestone: 1–4 seats → 5, 5–9 → 10, 10–19 → 20, and the chance of reaching it within a year; 20+ seats out of scope), `fit_dept_*` (enters a new department in 12 months), `depts_so_far*`, `fit_second_dept_*` (one-department clients, banded among them), `likely_next_dept*`, `seats_change_6m_net`, and `fit_growth_*` / `fit_reach_10_*` for clients under a year |
| New Logo | `industry` + `\|` + country from `region` (`propensity_country()`; blanks = Unknown) | `fit_growth_*`, `fit_reach_10_*`, `fit_branch_1y_*`, `likely_first_new_dept`, `fit_outlook` |

Only the score columns are taken: the dashboard keeps its own `industry`, `region`, `shortcode` and `crm_company_id`.
`fit_coverage` says why a row has no score (`not in propensity extract` for Organic Growth rows the extract doesn't have).

**What each score means** comes with the extract (`score_definitions`): the question it answers, who it applies to, the exact outcome and time window, the assumptions (New Logo: *if we sign them*, scored by industry and country only), the historical average, AUC and band cut-offs. It is saved in the dashboard JSON (`meta.propensity.definitions`) and in the output workbook (`propensity_definitions` sheet).

In [ ]:
NOT_IN_EXTRACT = "not in propensity extract"
PROPENSITY_NUMERIC = [c for c in PROPENSITY_COLS if c.endswith(("_p", "_vs_peers"))
                      or c in ("seats_change_6m_net", "fit_seat_milestone", "depts_so_far")]
PROPENSITY_BAND_VALUES = ["Low", "Medium", "High"]
PROFILE_COLUMNS = {   # prospect_profiles column -> output column (New Logo)
    "fit_growth_p": "fit_growth_p", "fit_growth_band": "fit_growth_band", "fit_growth_why": "fit_growth_why",
    "fit_reach_10_p": "fit_reach_10_p", "fit_reach_10_band": "fit_reach_10_band", "fit_reach_10_why": "fit_reach_10_why",
    "fit_branch_1y_p": "fit_branch_1y_p", "fit_branch_1y_band": "fit_branch_1y_band", "fit_branch_1y_why": "fit_branch_1y_why",
    "likely_first_new_dept": "likely_first_new_dept", "outlook": "fit_outlook",
}

def propensity_country(region):
    """Region code -> the country level the propensity models use (Client Propensity's region_to_country)."""
    if not isinstance(region, str) or not region.strip():
        return "Unknown"
    return "Australia" if region.startswith("AU") else PROPENSITY_COUNTRY.get(region, "Other")

def extract_propensity(data_dir=DATA_DIR):
    """The newest Client Propensity extract in data_dir: accounts, prospect profiles, and run info."""
    files = sorted(data_dir.glob(PROPENSITY_ACCOUNTS_GLOB))
    if not files:
        raise FileNotFoundError(f"No {PROPENSITY_ACCOUNTS_GLOB} in {data_dir}. Run Client Propensity/run_pipeline.py "
                                "(it copies its extract here), or set PROPENSITY_MODE = 'mock' or 'off'.")
    acc_path = files[-1]
    stem = acc_path.name[: -len("_propensity_accounts.csv")]
    prof_path, xlsx = data_dir / f"{stem}_prospect_profiles.csv", data_dir / f"{stem}.xlsx"
    acc = pd.read_csv(acc_path, dtype={"account_id": str, "shortcode": str, "crm_company_id": str})
    prof = pd.read_csv(prof_path, dtype={"profile_key": str})
    run_info = (pd.read_excel(xlsx, sheet_name="run_info", dtype=str).set_index("item")["value"].to_dict()
                if xlsx.exists() else {})
    as_of = run_info.get("as_of", stem[:10])
    info = {"mode": "real", "as_of": as_of, "model_version": run_info.get("model_version", "?"),
            "accounts_file": acc_path.name, "profiles_file": prof_path.name, "run_info": run_info,
            "definitions": read_propensity_definitions(data_dir / f"{stem}_score_definitions.csv"),
            "source": f"Client Propensity extract {as_of} ({run_info.get('model_version', '?')})"}
    return acc, prof, info

def read_propensity_definitions(path):
    """What each score means (question, who, outcome, window, assumptions, base rate, AUC, band cut-offs), as written by
    Client Propensity. [] if the extract predates the definitions file."""
    if not path.exists():
        return []
    d = pd.read_csv(path).astype(object)
    return d.where(d.notna(), None).to_dict("records")      # blanks -> None (strict JSON)

def newest_propensity_definitions(data_dir=DATA_DIR):
    files = sorted(data_dir.glob("*_Propensity_Extract_score_definitions.csv"))
    return read_propensity_definitions(files[-1]) if files else []

def _real_propensity(out):
    acc, prof, info = extract_propensity()
    vals = pd.DataFrame(index=out.index, columns=PROPENSITY_COLS, dtype=object)
    og, nl = out["goal"] == GOALS["mystaff"], out["goal"] == GOALS["hubspot"]
    # Organic Growth: one-to-one on account_id; only the score columns (the dashboard keeps its own keys and descriptions)
    take = [c for c in PROPENSITY_COLS if c in acc.columns]
    m = (out.loc[og, ["account_id"]].reset_index()
            .merge(acc[["account_id"] + take], on="account_id", how="left", validate="one_to_one")
            .set_index("index"))
    vals.loc[og, take] = m[take]
    vals.loc[og, "fit_coverage"] = vals.loc[og, "fit_coverage"].where(vals.loc[og, "fit_coverage"].notna(), NOT_IN_EXTRACT)
    # New Logo: prospect profile for industry|country
    key = out.loc[nl, "industry"].fillna("Unknown").astype(str) + "|" + out.loc[nl, "region"].map(propensity_country)
    p = prof.set_index("profile_key")
    for src, dst in PROFILE_COLUMNS.items():
        if src in p:
            vals.loc[nl, dst] = key.map(p[src])
    vals.loc[nl, "fit_coverage"] = np.where(key.isin(p.index), "prospect profile: " + key, "no prospect profile")
    return vals, info

# ---- mock: synthetic values, seeded per account_id ------------------------------------------------------------
MOCK_PROPENSITY_DEPTS = ["Accounting", "Virtual Assistant", "BPO Voice", "IT Software Development", "Administrative",
                         "Real Estate", "Sales Professional", "Creative, Design & Multimedia", "Marketing/Content",
                         "Data Science / Analytics", "Legal Specialist", "BPO Non-Voice/Back-Office"]
MOCK_PROPENSITY_CUTS = {"fit_seat_milestone": (0.05, 0.20), "fit_dept_expand": (0.13, 0.22), "fit_growth": (0.05, 0.08),
                        "fit_reach_10": (0.02, 0.03), "fit_branch_1y": (0.21, 0.29), "fit_second_dept": (0.11, 0.17)}   # Low / Medium / High cut-offs (like the real ones)

def _band(p, cuts):
    return "Low" if p < cuts[0] else "Medium" if p < cuts[1] else "High"

def _outlook(growth_band, branch_band):
    if growth_band == "High" and branch_band == "High":
        return "Grow + expand"
    if growth_band == "High":
        return "Likely to grow"
    if branch_band == "High":
        return "Likely to expand"
    return "Low propensity" if growth_band == branch_band == "Low" else "Moderate"

def _mock_propensity(goal, rng):
    """One account's synthetic propensity values (FICTIONAL)."""
    r = {}
    def score(name, a, b):
        p = rng.betavariate(a, b)
        r[f"{name}_p"], r[f"{name}_band"] = round(p, 3), _band(p, MOCK_PROPENSITY_CUTS[name])
    def depts(k):
        return ", ".join(f"{d} {rng.randint(6, 20)}% ({rng.uniform(0.5, 2.5):.1f}x)" for d in rng.sample(MOCK_PROPENSITY_DEPTS, k))
    if goal == GOALS["mystaff"]:
        u = rng.random()
        if u < 0.25:
            r["fit_coverage"] = "no current seats"
            return r
        if u < 0.30:
            r["fit_coverage"] = NOT_IN_EXTRACT
            return r
        r["fit_coverage"] = "scored"
        r["fit_seat_milestone"] = rng.choices([5, 10, 20, None], weights=[0.74, 0.14, 0.07, 0.05])[0]   # None = 20+ seats
        if r["fit_seat_milestone"] is not None:
            score("fit_seat_milestone", 1.2, 8.0)
            r["fit_seat_milestone_why"] = "mock"
        score("fit_dept_expand", 2.0, 7.5)
        r["fit_dept_vs_peers"] = round(rng.lognormvariate(0, 0.35), 2)
        r["fit_dept_why"] = "mock"
        r["depts_so_far"] = rng.choices([1, 2, 3, 4], weights=[0.45, 0.2, 0.15, 0.2])[0]
        r["depts_so_far_names"] = "; ".join(rng.sample(MOCK_PROPENSITY_DEPTS, r["depts_so_far"]))
        if r["depts_so_far"] == 1:
            r["fit_second_dept_p"] = r["fit_dept_expand_p"]
            r["fit_second_dept_band"] = _band(r["fit_dept_expand_p"], MOCK_PROPENSITY_CUTS["fit_second_dept"])
        r["likely_next_dept"], r["likely_next_dept_by_lift"] = depts(2), depts(2)
        r["seats_change_6m_net"] = int(round(rng.gauss(0.4, 2.0)))
        if rng.random() < 0.10:                              # clients under a year also get new-client growth scores
            score("fit_growth", 1.6, 16)
            score("fit_reach_10", 1.2, 25)
            r["fit_growth_why"] = r["fit_reach_10_why"] = "mock"
    else:
        score("fit_growth", 1.6, 16)
        score("fit_reach_10", 1.2, 25)
        score("fit_branch_1y", 3.0, 8.0)
        r["fit_growth_why"] = r["fit_reach_10_why"] = r["fit_branch_1y_why"] = "mock"
        r["likely_first_new_dept"] = depts(2)
        r["fit_outlook"] = _outlook(r["fit_growth_band"], r["fit_branch_1y_band"])
        r["fit_coverage"] = "prospect profile (mock)"
    return r

def generate_mock_propensity(out, seed=MOCK_SEED):
    """SYNTHETIC propensity values, seeded per account_id (same values every run, independent of row order)."""
    rows = [_mock_propensity(g, random.Random(f"{seed}:propensity:{a}")) for a, g in zip(out["account_id"], out["goal"])]
    vals = pd.DataFrame(rows, index=out.index).reindex(columns=PROPENSITY_COLS).astype(object)
    info = {"mode": "mock", "as_of": SCORING_AS_OF.strftime("%Y-%m-%d"), "model_version": "mock",
            "source": f"MOCK propensity (synthetic, seed={seed})",
            # the meaning of each score is the real one; the values are not
            "definitions": [{**d, "base_rate": None, "auc": None, "band_cutoffs": None,
                             "assumptions": "MOCK VALUES. " + str(d.get("assumptions", ""))}
                            for d in newest_propensity_definitions()],
            "run_info": {f"band_cutoffs_{k}": f"{a:.3f} / {b:.3f}" for k, (a, b) in MOCK_PROPENSITY_CUTS.items()}}
    return vals, info

def add_propensity(out, mode=PROPENSITY_MODE):
    """Stage 7b: fill the 5_propensity group. Returns (out, info); info feeds the DQ checks, the dashboard meta and run_info."""
    out = out.copy()
    if mode == "real":
        vals, info = _real_propensity(out)
    elif mode == "mock":
        vals, info = generate_mock_propensity(out)
    elif mode == "off":
        vals = pd.DataFrame(index=out.index, columns=PROPENSITY_COLS, dtype=object)
        info = {"mode": "off", "source": "off (PROPENSITY_MODE = 'off')", "run_info": {}, "definitions": []}
    else:
        raise ValueError(f"PROPENSITY_MODE must be 'real', 'mock' or 'off', not {mode!r}")
    for c in PROPENSITY_COLS:
        if c == "propensity_source":
            continue
        out[c] = pd.to_numeric(vals[c], errors="coerce") if c in PROPENSITY_NUMERIC else vals[c].where(vals[c].notna(), None)
    out["propensity_source"] = info["source"]
    return out, info

def run_propensity_checks(out, info):
    rows = []
    def check(name, level, ok, detail=""):
        rows.append({"check": name, "level": level, "passed": bool(ok), "detail": detail})
    og, nl = out["goal"] == GOALS["mystaff"], out["goal"] == GOALS["hubspot"]
    check("every row tagged with propensity_source", "error", out["propensity_source"].notna().all())
    if info["mode"] == "off":
        check("propensity signals filled", "info", False, "PROPENSITY_MODE = 'off'")
        return pd.DataFrame(rows)
    p_cols = [c for c in PROPENSITY_COLS if c.endswith("_p")]
    bad_p = [c for c in p_cols if not out[c].dropna().between(0, 1).all()]
    check("fit_*_p within 0-1", "error", not bad_p, f"out of range: {bad_p}")
    band_cols = [c for c in PROPENSITY_COLS if c.endswith("_band")]
    bad_b = [c for c in band_cols if not out[c].dropna().isin(PROPENSITY_BAND_VALUES).all()]
    check("bands are Low / Medium / High", "error", not bad_b, f"invalid: {bad_b}")
    check("Organic Growth only: fit_seat_milestone* / fit_dept_*; New Logo only: fit_branch_1y_*", "error",
          out.loc[nl, ["fit_seat_milestone_p", "fit_dept_expand_p"]].isna().all().all()
          and out.loc[og, "fit_branch_1y_p"].isna().all())
    matched = (out.loc[og, "fit_coverage"] != NOT_IN_EXTRACT).mean()
    check(f"Organic Growth rows found in the propensity extract (>= {PROPENSITY_MIN_MATCH:.0%})", "warn",
          matched >= PROPENSITY_MIN_MATCH, f"{matched:.1%} ({int((out.loc[og, 'fit_coverage'] == NOT_IN_EXTRACT).sum())} not found)")
    scored = out.loc[og, "fit_dept_expand_p"].notna().mean()
    check(f"Organic Growth rows with a fit score (>= {PROPENSITY_MIN_SCORED:.0%})", "warn",
          scored >= PROPENSITY_MIN_SCORED, f"{scored:.1%}")
    profiled = out.loc[nl, "fit_growth_p"].notna().mean()
    check("every New Logo row has a prospect profile", "warn", profiled == 1, f"{profiled:.1%}")
    if info["mode"] == "real":
        age = abs((SCORING_AS_OF - pd.Timestamp(info["as_of"])).days)
        check(f"propensity extract within {PROPENSITY_MAX_AGE_DAYS} days of SCORING_AS_OF", "warn",
              age <= PROPENSITY_MAX_AGE_DAYS, f"extract as of {info['as_of']} ({age} days); {info['run_info'].get('source_mystaff', '')}")
    second = out["fit_second_dept_band"].notna()
    check("fit_second_dept_* only for one-department Organic Growth clients", "error",
          (out.loc[second, "goal"] == GOALS["mystaff"]).all() and (out.loc[second, "depts_so_far"] == 1).all())
    ms = out.loc[og, "fit_seat_milestone"].dropna()
    check("seat milestones are 5 / 10 / 20, and only for clients below 20 seats", "error",
          ms.isin([5, 10, 20]).all() and out.loc[og & out["fit_seat_milestone"].isna(), "fit_seat_milestone_p"].isna().all())
    check("propensity uses real data", "warn", info["mode"] == "real",
          "propensity signals are SYNTHETIC (mock)" if info["mode"] == "mock" else "")
    return pd.DataFrame(rows)

def propensity_meta(info):
    """What the dashboard shows about the propensity scores (strict-JSON friendly)."""
    if not info or info.get("mode") == "off":
        return {"mode": "off"}
    ri = info.get("run_info", {})
    return {"mode": info["mode"], "source": info["source"], "as_of": info.get("as_of"),
            "model_version": info.get("model_version"),
            "auc": {k[4:]: float(v) for k, v in ri.items() if k.startswith("auc") and v not in (None, "")},
            "band_cutoffs": {k[len("band_cutoffs_"):]: v for k, v in ri.items() if k.startswith("band_cutoffs_")},
            # what each score means: question, who it applies to, outcome, window, assumptions, base rate, AUC
            "definitions": info.get("definitions", [])}

def propensity_notice(info):
    """Sentence added to the dashboard's warning banner."""
    if not info or info.get("mode") == "off":
        return ""
    if info["mode"] == "mock":
        return " <b>Propensity (fit) scores are MOCK values made up for testing.</b>"
    return f" Propensity (fit) scores are real: {info['source']}."

In [ ]:
output, propensity_info = add_propensity(output)
dq_propensity = run_propensity_checks(output, propensity_info)
display(dq_propensity)
assert_dq(dq_propensity)
dq = pd.concat([dq, dq_propensity], ignore_index=True)
print(propensity_info["source"])
pd.crosstab(output["goal"], output["fit_coverage"].fillna("(blank)").str.split(":").str[0], margins=True)

In [ ]:
# INSPECT: fit bands by goal, and a few rows
display(pd.concat({c: output.groupby("goal")[c].value_counts().unstack(fill_value=0)
                   for c in ["fit_dept_expand_band", "fit_growth_band"]}, names=["score"]))
display(pd.crosstab(output["fit_seat_milestone"].fillna(0).astype(int).replace(0, "none (20+ seats / not scored)"),
                    output["fit_seat_milestone_band"].fillna("(blank)"), margins=True))
output.loc[output["fit_coverage"].eq("scored"), ["account_id", "company_name", "cs_staff_current", "fit_seat_milestone",
                                                 "fit_seat_milestone_band", "fit_dept_expand_band",
                                                 "likely_next_dept_by_lift", "fit_dept_why"]].head(8)

## 8. Score (`score`)
Fills group **3_scoring** (except routing). The formulas are the mock workbook's. The settings are `SCORING`, the live
**`dashboard_v3`** profile: `dashboard_v2` (below) plus a **Fit** component from the Client Propensity bands (stage 7b).

| Max points | Website | News / jobs | Seat growth | Diamond / Competitor | **Fit** |
|---|---|---|---|---|---|
| New Logo | 30 | 30 (news 15, hiring 15) | – | 10 / 10 | **20**: profile growth (better of 5+ in 1y / 10+ in 2y) 15, second department 5 |
| Organic Growth | 30 | 30 (news 15, hiring 15) | 15 | – | **25**: next seat milestone 0.6, second department 0.4 (one-department clients only; 2+ departments score 0 on it, so at most 15 points) |

Each fit part is its band: High 1, Medium 0.5, Low 0 (`FIT`). A part without a score leaves the other on its own; no fit at
all = unknown, scored at the goal's average fit and listed in `estimated_fields` ("Fit"). Details: BUSINESS_RULES §11.2b.

`dashboard_v2`, and how it differs from the workbook and from `dashboard`:

| Setting | Workbook | `dashboard` (until v0.2) | **`dashboard_v2`** (v0.3; still holds in `dashboard_v3` except the points) |
|---|---|---|---|
| Max points, New Logo | website 35, news/jobs 35, Diamond 15, prior BPO 15 | website 40, news/jobs 40, Diamond 10, **Competitor client** 10 | same as `dashboard` |
| Max points, Organic Growth | website 50, news/jobs 50 | website 40, news/jobs 40, **seat growth 20** | same as `dashboard` |
| Website signal shares | Team Builder 0.4, form fill 0.3, volume 0.3 | 0.5 / 0.25 / 0.25 | same |
| News/jobs signal shares | news 0.6, hiring 0.4 | 0.625 / 0.375 | same |
| Seat growth bands (seats added in ~6 months) | n/a | 0 → 0%, 1–5 → 25%, 6–10 → 50%, 11+ → 100% | same |
| **Unknown component** | website 35%, news/jobs 30% of max | **25%** of max for every component | **the average signal of the same goal's accounts where it is known** (25% only if none is known) |
| **Act now triggers** | Diamond referral, form fill, a client's Team Builder visit, funding round / PE buyout in 30 days | the four, plus a prospect's Team Builder visit, hiring spike or 18+ job postings (New Logo), **10+** visits with no form fill, restructuring / leadership-change news in **60** days | the same, with **20+** visits and a **30**-day window for restructuring / leadership news |
| **Act now bonus** | none | +15 (+7.5 Diamond) for the **first** trigger | **every** trigger family adds: the best bonus per family (Website, News, Hiring, Referral), sorted, each further family × 0.5 (15 → +7.5 → +3.75) |
| **Tier** | Tier 1 ≥ 70, Tier 2 ≥ 40 | same | **by rank within each goal**: top 10% Tier 1, next 30% Tier 2 (ties share the higher tier) |
| **Order (`sort_key`)** | score + 1000 if hot (Act now first) | same | **score only**: the bonus lifts Act now accounts, but doesn't pin them |

`hot_trigger` is still the **first** trigger that applies (Diamond referral → form fill → Team Builder visit → funding round /
PE buyout → hiring spike → heavy website visits → restructuring / leadership change); it decides the owner team (§9).
`act_now_triggers` lists every trigger that applies.

Two golden tests check the scoring code (the old behaviour is still there for the profiles that don't set the new keys):
1. **Workbook** settings must reproduce the 400 mock accounts in `reference materials/data/accounts.csv`.
2. **Dashboard reference** settings (the `dashboard` profile without the extended triggers) must reproduce the 800 accounts in
   `data/account_priority_dashboard.json` (the dashboard's own output).


In [45]:
def xl_round(x, digits=0):
    """Excel ROUND: half away from zero (Python's round() is half-to-even)."""
    m = 10 ** digits
    return math.copysign(math.floor(abs(x) * m + 0.5 + 1e-9) / m, x)

def _blank(v):
    return v is None or (not isinstance(v, str) and pd.isna(v)) or v == ""

def _num(v):
    return 0 if _blank(v) else float(v)

def _recency(days):
    return next(f for lo, f in reversed(RECENCY) if days >= lo)

def _seat_share(seats, bands):
    return next(share for lo, share in reversed(bands) if seats >= lo)

def _band_value(band, F):
    return None if _blank(band) else F["band_value"].get(band)

def _fit_signal(r, S):
    """One account's 0-1 fit signal from its Client Propensity bands (S["fit"], see FIT); None = unknown (no fit score)."""
    F = S.get("fit")
    if not F or not S["max_points"][r["goal"]].get("fit", 0):
        return None
    if r["goal"] == GOALS["mystaff"]:
        W = F["Organic Growth"]
        # department part: one-department clients only (do they grow past one department?). 2+ departments score
        # multi_dept_value (0) on it with the weights kept, so their fit is at most seat_weight
        seat = _band_value(r.get("fit_seat_milestone_band"), F)
        n = r.get("depts_so_far")
        if not _blank(n) and n > 1:
            return None if seat is None else round(seat * W["seat_weight"] + W["multi_dept_value"] * W["dept_weight"], 3)
        parts = [(seat, W["seat_weight"]), (_band_value(r.get("fit_second_dept_band"), F), W["dept_weight"])]
    else:
        W = F["New Logo"]
        growth = [v for v in (_band_value(r.get("fit_growth_band"), F), _band_value(r.get("fit_reach_10_band"), F)) if v is not None]
        parts = [(max(growth) if growth else None, W["growth_weight"]), (_band_value(r.get("fit_branch_1y_band"), F), W["dept_weight"])]
    known = [(v, w) for v, w in parts if v is not None]
    return round(sum(v * w for v, w in known) / sum(w for _, w in known), 3) if known else None

def _signals(r, as_of, S):
    """One account's 0-1 signals (website, news/jobs, seat growth, fit); None = unknown."""
    W, N = S["web"], S["news"]
    rnd_sig = (lambda x: xl_round(x, S["round_signals"])) if S["round_signals"] is not None else (lambda x: x)
    web = None if _blank(r["web_visits_30d"]) else rnd_sig(
        (W["tb_share"] if r["team_builder_visit"] == "Yes" else 0)
        + (W["ff_share"] if _num(r["form_fill_30d"]) > 0 else 0)
        + min(float(r["web_visits_30d"]) / W["vol_cap"], 1) * W["vol_share"])
    news, ndate = r["news_signal"], r["news_signal_date"]
    if _blank(news) and _blank(r["job_postings_30d"]):
        nj = None
    else:
        strength = 0 if _blank(news) else NEWS_STRENGTH.get(news, NEWS_STRENGTH[NEWS_UNMAPPED])
        rec = 1 if _blank(news) or _blank(ndate) else _recency((as_of - ndate).days)
        hire = 1 if r["hiring_spike"] == "Yes" else min(_num(r["job_postings_30d"]) / N["jobs_cap"], 1)
        nj = rnd_sig(N["news_share"] * strength * rec + N["jobs_share"] * hire)
    seat_max = S["max_points"][r["goal"]].get("seat", 0)
    seats = r.get("seats_added_6m")
    seat = None if not seat_max or _blank(seats) else _seat_share(float(seats), S["seat_bands"])
    return web, nj, seat, _fit_signal(r, S)

def known_mean_fill(recs, as_of, S):
    """Share of max used for an unknown component, per goal and component: the average signal of the accounts of that
    goal where the component is known (S["baseline"] when none is known). Rounded to 3 decimals."""
    sig = pd.DataFrame([dict(zip(("web", "news", "seat", "fit"), _signals(r, as_of, S)), goal=r["goal"]) for r in recs])
    fill = {}
    for goal, d in sig.groupby("goal"):
        fill[goal] = {}
        for comp in ("web", "news", "seat", "fit"):
            known = pd.to_numeric(d[comp], errors="coerce").dropna()
            fill[goal][comp] = round(float(known.mean()), 3) if len(known) else S["baseline"].get(comp, 0.25)
    return fill

def act_now_bonus(triggers, A):
    """Bonus points for an account's Act now triggers (A = S["act_now_bonus"]). Without "stack_decay" only the first
    trigger counts (the dashboard profile); with it, the best bonus per trigger family, sorted, each further family
    worth stack_decay x the one before."""
    if not triggers or not A:
        return 0
    if A.get("stack_decay") is None:
        return A.get(triggers[0], A["default"])
    best = {}
    for t in triggers:
        fam = TRIGGER_FAMILY.get(t, t)
        best[fam] = max(best.get(fam, 0), A.get(t, A["default"]))
    return sum(b * A["stack_decay"] ** i for i, b in enumerate(sorted(best.values(), reverse=True)))

def score_account(r, as_of, S, fill=None):
    """One account through the scoring formulas with settings profile S. fill = share of max for an unknown component
    ({"web", "news", "seat", "fit"}); None = the profile's flat baseline."""
    goal = r["goal"]
    nl = goal == "New Logo"
    mx, H = S["max_points"][goal], S["hot"]
    F = fill or S["baseline"]

    # industry modifier
    trend = r["industry_trend"]
    mod = 1.0 if _blank(trend) else xl_round(S["modifier"]["at_trend_0"]
                                             + trend * (S["modifier"]["at_trend_1"] - S["modifier"]["at_trend_0"]), 3)
    # signals (0-1); None = unknown
    web, nj, seat, fit = _signals(r, as_of, S)
    seat_max, fit_max = mx.get("seat", 0), mx.get("fit", 0)

    # points (unknown -> fill share of max)
    web_pts = xl_round(mx["web"] * (F["web"] if web is None else web), 1)
    news_pts = xl_round(mx["news"] * (F["news"] if nj is None else nj), 1)
    seat_pts = xl_round(seat_max * (F["seat"] if seat is None else seat), 1) if seat_max else None
    fit_pts = xl_round(fit_max * (F.get("fit", S["baseline"].get("fit", 0.25)) if fit is None else fit), 1) if fit_max else None
    dia = (mx["diamond"] if r["diamond_referral"] == "Yes" else 0) if nl else "N/A"
    bpo = (mx["bpo"] if r["prior_bpo_flag"] == "Yes" else 0) if nl else "N/A"
    base = xl_round(web_pts + news_pts + (dia + bpo if nl else 0) + (seat_pts or 0) + (fit_pts or 0), 1)
    raw = base * mod
    score_before = int(xl_round(min(raw, 100), 0))

    # Act now triggers: every one that applies, in priority order; the first is hot_trigger (it decides the team)
    news, ndate = r["news_signal"], r["news_signal_date"]
    news_age = None if _blank(news) or _blank(ndate) else (as_of - ndate).days
    top_news = (H["top_news"] and news_age is not None and NEWS_STRENGTH.get(news, 0) >= H["top_news_min_strength"]
                and news_age <= H["top_news_days"])
    hiring = nl and H["hiring_spike"] and (r["hiring_spike"] == "Yes" or _num(r["job_postings_30d"]) >= H["hiring_min_postings"])
    heavy = (H["heavy_visits_min"] is not None and _num(r["web_visits_30d"]) >= H["heavy_visits_min"]
             and not _num(r["form_fill_30d"]) > 0)
    campaign = news_age is not None and news in H["campaign_news"] and news_age <= H["campaign_news_days"]
    triggers = []
    if nl and r["diamond_referral"] == "Yes" and H["diamond"]:
        triggers.append("Diamond referral")
    if _num(r["form_fill_30d"]) > 0 and H["form_fill"]:
        triggers.append("Form fill")
    if r["team_builder_visit"] == "Yes" and (H["team_builder"] if not nl else H["team_builder_prospect"]):
        triggers.append("Team Builder visit")                  # a client's visit (Organic Growth) or a prospect's (New Logo)
    if top_news:
        triggers.append(news)                                  # Funding round / PE buyout in the last 30 days
    if hiring:
        triggers.append("Hiring spike")
    if heavy:
        triggers.append("Heavy website visits")                # Marketing trigger
    if campaign:
        triggers.append(news)                                  # Restructuring / Leadership change: Marketing trigger
    hot = triggers[0] if triggers else None

    # Act now bonus, added after the industry nudge; the score is capped at 100
    bonus = act_now_bonus(triggers, S["act_now_bonus"])
    score = int(xl_round(min(raw + bonus, 100), 0)) if bonus else score_before

    unknown = [("Website", web is None), ("News", nj is None), ("Seat growth", bool(seat_max) and seat is None),
               ("Fit", bool(fit_max) and fit is None)]
    est = ", ".join(name for name, unk in unknown if unk) or None
    known = ((0 if web is None else mx["web"]) + (0 if nj is None else mx["news"])
             + (mx["diamond"] + mx["bpo"] if nl else 0) + (seat_max if seat is not None else 0) + (fit_max if fit is not None else 0))
    T = S["tier"]
    coverage = known / sum(mx.values())
    if T.get("method", "score") == "score":
        tier = ("Tier 1" if score >= T["tier1"] and (not T["cap_tier1_on_coverage"] or coverage >= T["tier1_min_coverage"])
                else "Tier 2" if score >= T["tier2"] else "Tier 3")
    else:
        tier = None                                            # percentile tiers need every account: set in score_accounts()
    return {
        "industry_modifier": mod, "web_signal (0-1)": web, "news_jobs_signal (0-1)": nj, "seat_signal (0-1)": seat,
        "web_pts": web_pts, "news_jobs_pts": news_pts, "seat_pts": seat_pts, "diamond_pts": dia, "bpo_pts": bpo,
        "fit_signal (0-1)": fit, "fit_pts": fit_pts,
        "base_points": base, "score_before_bonus": score_before, "act_now_bonus": bonus, "priority_score": score,
        "estimated_fields": est, "data_coverage": round(coverage, 4), "hot_trigger": hot,
        "act_now_triggers": "; ".join(triggers) or None, "tier": tier,
        "sort_key": score + (1000 if hot and S.get("hot_first", True) else 0),
        # the workbook's own routing (Organic Growth -> AM), only for the workbook golden test
        "_workbook_team": "AM" if not nl else ("BDM" if (r["diamond_referral"] == "Yes" or hot or score >= BDM_MIN_SCORE) else "SDR"),
    }

def percentile_tier(scores, goals, S=SCORING):
    """Tier by rank within each goal: the top tier1_share of a goal's accounts are Tier 1, the next tier2_share Tier 2.
    An account's position = share of its goal's accounts that score strictly higher, so ties share the higher tier."""
    T = S["tier"]
    scores, goals = pd.Series(scores), pd.Series(goals, index=pd.Series(scores).index)
    above = (scores.groupby(goals).rank(method="min", ascending=False) - 1) / goals.map(goals.value_counts())
    return pd.Series(np.select([above < T["tier1_share"], above < T["tier1_share"] + T["tier2_share"]],
                               ["Tier 1", "Tier 2"], "Tier 3"), index=scores.index)

def tiers(scores, goals, S=SCORING):
    """Tier for a set of scores (e.g. the score before the bonus) with profile S's tier rule."""
    T = S["tier"]
    if T.get("method", "score") == "percentile":
        return percentile_tier(scores, goals, S)
    return pd.Series(scores).map(lambda v: "Tier 1" if v >= T["tier1"] else "Tier 2" if v >= T["tier2"] else "Tier 3")

def score_accounts(df, as_of, S):
    recs = df.to_dict("records")
    fill = known_mean_fill(recs, as_of, S) if S.get("unknown_fill") == "known_mean" else {}
    out = pd.DataFrame([score_account(r, as_of, S, fill.get(r["goal"])) for r in recs], index=df.index)
    if S["tier"].get("method", "score") == "percentile":
        out["tier"] = percentile_tier(out["priority_score"], df["goal"], S)
    return out

def score(out, S=SCORING, as_of=SCORING_AS_OF):
    """Stage 8: fill the 3_scoring group (routing columns come in stage 9)."""
    out = out.copy()
    scored = score_accounts(out, as_of, S)
    for c in SCORING_COLS:
        if c in scored:
            out[c] = scored[c]
    return out

In [46]:
# TEST helpers: compare our scores with a reference, column by column
TEXT_COLS = {"estimated_fields", "hot_trigger", "tier", "assigned_team"}

def _same(a, b, col):
    if col in TEXT_COLS:
        return a.fillna("").astype(str).values == b.fillna("").astype(str).values
    num = lambda s: pd.to_numeric(pd.Series(s).replace("N/A", np.nan), errors="coerce").astype(float).values
    return np.isclose(num(a), num(b), atol=1e-6, equal_nan=True)

def golden_check(ours: dict, ref: pd.DataFrame, label):
    res = pd.DataFrame({col: {"mismatches": int((~_same(o, ref[col], col)).sum())} for col, o in ours.items()}).T
    display(res)
    assert res["mismatches"].sum() == 0, f"{label}: scoring does not reproduce the reference"
    print(f"{label}: {len(ref)} accounts, {len(ours)} columns, 0 mismatches.")

In [47]:
# TEST 1 (regression): WORKBOOK settings must reproduce the mock workbook's 400 scored accounts
# keep_default_na=False: the news category "None" (= checked, nothing found) must not be read as missing
golden = pd.read_csv(GOLDEN_PATH, dtype={"employee_band": str}, keep_default_na=False, na_values=[""])
g_in = golden[COMPANY_COLS + MOCK_SIGNAL_COLS].copy()
g_in["news_signal_date"] = pd.to_datetime(g_in["news_signal_date"])
g_in["industry_trend"] = g_in["industry"].map(GOLDEN_INDUSTRY_TREND)
g_out = score_accounts(g_in, GOLDEN_AS_OF, SCORING_WORKBOOK)
golden_check({
    "industry_modifier": g_out["industry_modifier"], "web_signal": g_out["web_signal (0-1)"],
    "news_jobs_signal": g_out["news_jobs_signal (0-1)"], "web_pts": g_out["web_pts"], "news_jobs_pts": g_out["news_jobs_pts"],
    "diamond_pts": g_out["diamond_pts"], "bpo_pts": g_out["bpo_pts"], "base_points": g_out["base_points"],
    "priority_score": g_out["priority_score"], "estimated_fields": g_out["estimated_fields"],
    "data_coverage": g_out["data_coverage"], "hot_trigger": g_out["hot_trigger"], "tier": g_out["tier"],
    "assigned_team": g_out["_workbook_team"], "sort_key": g_out["sort_key"],
}, golden, "Workbook golden test")

,mismatches
industry_modifier,0
web_signal,0
news_jobs_signal,0
web_pts,0
news_jobs_pts,0
diamond_pts,0
bpo_pts,0
base_points,0
priority_score,0
estimated_fields,0


Workbook golden test: 400 accounts, 15 columns, 0 mismatches.


In [48]:
# TEST 2: DASHBOARD REFERENCE settings (the live profile without the extended triggers) must reproduce the dashboard's 800 scored accounts (prep.py output)
dash = pd.DataFrame(json.load(open(DASHBOARD_GOLDEN_PATH, encoding="utf-8"))["rows"])
d_in = dash.rename(columns={"competitor_client": "prior_bpo_flag"})[["goal", "industry_trend", "seats_added_6m"] + MOCK_SIGNAL_COLS].copy()
d_in["news_signal_date"] = pd.to_datetime(d_in["news_signal_date"])
# prep.py read accounts.csv with pandas' default NA values, which turned the news category "None" (checked, nothing found)
# into blank. The dashboard still treats those accounts' news as known (from estimated_fields), so restore "None" here.
news_known = ~dash["estimated_fields"].fillna("").str.contains("News")
d_in.loc[news_known & d_in["news_signal"].isna(), "news_signal"] = "None"
d_out = score_accounts(d_in, GOLDEN_AS_OF, SCORING_DASHBOARD_REFERENCE)
golden_check({
    "web_pts": d_out["web_pts"], "news_jobs_pts": d_out["news_jobs_pts"], "seat_pts": d_out["seat_pts"],
    "base_points": d_out["base_points"], "workbook_score": d_out["score_before_bonus"],
    "priority_score": d_out["priority_score"], "tier": d_out["tier"], "hot_trigger": d_out["hot_trigger"],
    "estimated_fields": d_out["estimated_fields"], "sort_key": d_out["sort_key"],
}, dash, "Dashboard golden test")

,mismatches
web_pts,0
news_jobs_pts,0
seat_pts,0
base_points,0
workbook_score,0
priority_score,0
tier,0
hot_trigger,0
estimated_fields,0
sort_key,0


Dashboard golden test: 800 accounts, 10 columns, 0 mismatches.


In [49]:
output = score(output)
output[["account_id", "goal", "priority_score", "tier", "hot_trigger", "act_now_bonus", "score_before_bonus",
        "web_pts", "news_jobs_pts", "seat_pts", "fit_pts", "diamond_pts", "bpo_pts", "estimated_fields"]].head(15)

,account_id,goal,priority_score,tier,hot_trigger,act_now_bonus,score_before_bonus,web_pts,news_jobs_pts,seat_pts,diamond_pts,bpo_pts,estimated_fields
0,HS-4177537874,New Logo,20,Tier 3,NaN,0.00,20,10.4,8.8,NaN,0,0,Website
1,HS-9181185458,New Logo,41,Tier 2,Diamond referral,7.50,34,10.4,11.3,NaN,10,0,"Website, News"
2,HS-16740821091,New Logo,23,Tier 3,NaN,0.00,23,10.4,11.3,NaN,0,0,"Website, News"
3,HS-57844246813,New Logo,65,Tier 1,Form fill,15.00,50,38.0,11.3,NaN,0,0,News
4,HS-4319585420,New Logo,32,Tier 2,NaN,0.00,32,10.4,11.3,NaN,0,10,"Website, News"
5,HS-8337617848,New Logo,54,Tier 2,Diamond referral,18.75,35,14.0,11.3,NaN,10,0,News
6,HS-4675979868,New Logo,18,Tier 3,NaN,0.00,18,6.0,11.3,NaN,0,0,News
7,HS-6947188532,New Logo,12,Tier 3,NaN,0.00,12,10.4,1.5,NaN,0,0,Website
8,HS-57844247725,New Logo,23,Tier 3,NaN,0.00,23,10.4,11.3,NaN,0,0,"Website, News"
9,HS-57839701870,New Logo,17,Tier 3,NaN,0.00,17,6.5,10.0,NaN,0,0,NaN


In [50]:
# INSPECT: what the Act now bonus changes (tier before -> after; with percentile tiers "before" ranks the scores before the bonus)
before = tiers(output["score_before_bonus"], output["goal"])
pd.crosstab([output["goal"], before.rename("tier before bonus")], output["tier"].rename("tier after bonus"))


tier after bonus                  Tier 1  Tier 2  Tier 3
goal           tier before bonus                        
New Logo       Tier 1                 20       2       0
               Tier 2                  2      60       4
               Tier 3                  0       9     120
Organic Growth Tier 1                102       9       0
               Tier 2                 15     275      58
               Tier 3                  0      46     605

In [51]:
# INSPECT: what the Fit component changes (live dashboard_v3 vs the previous live profile, dashboard_v2). Mock signals: illustration only.
prev = score_accounts(output, SCORING_AS_OF, SCORING_DASHBOARD_V2)
cmp = pd.DataFrame({"goal": output["goal"], "prev_tier": prev["tier"], "new_tier": output["tier"],
                    "prev_hot": prev["hot_trigger"].notna(), "new_hot": output["hot_trigger"].notna(),
                    "n_triggers": output["act_now_triggers"].fillna("").map(lambda t: len(t.split("; ")) if t else 0)})
display(pd.concat({"previous (dashboard_v2)": pd.crosstab(cmp["goal"], cmp["prev_tier"]),
                   "live (dashboard_v3)": pd.crosstab(cmp["goal"], cmp["new_tier"])}, axis=1))
# fit points by goal, and how many accounts moved tier
display(output.groupby("goal")["fit_pts"].describe().round(1))
display(pd.crosstab([cmp["goal"], cmp["prev_tier"].rename("tier, dashboard_v2")], cmp["new_tier"].rename("tier, dashboard_v3")))
display(cmp.groupby("goal")[["prev_hot", "new_hot"]].mean().round(3)
           .rename(columns={"prev_hot": "share Act now, previous", "new_hot": "share Act now, live"}))
display(pd.crosstab(cmp["goal"], cmp["n_triggers"]).rename_axis(columns="Act now triggers per account"))
print("Unknown fill (share of max), live:", known_mean_fill(output.to_dict("records"), SCORING_AS_OF, SCORING))
# how much the top of each goal changes (rank by sort_key within goal, as the lists are ordered)
rank = lambda s: s.groupby(output["goal"]).rank(method="first", ascending=False)
top = pd.DataFrame({"prev": rank(prev["sort_key"]), "new": rank(output["sort_key"]), "goal": output["goal"]})
pd.DataFrame({n: {g: int(((d["prev"] <= n) & (d["new"] <= n)).sum()) for g, d in top.groupby("goal")} for n in (25, 50, 100)}) \
  .rename(columns=lambda n: f"in both top {n}")


previous (dashboard)               live (dashboard_v2)              
prev_tier                    Tier 1 Tier 2 Tier 3              Tier 1 Tier 2 Tier 3
goal                                                                               
New Logo                          5     49    163                  22     71    124
Organic Growth                   49    355    706                 117    330    663

,"share Act now, previous","share Act now, live"
goal,,
New Logo,0.392,0.327
Organic Growth,0.487,0.339


Act now triggers per account,0,1,2,3
goal,,,,
New Logo,146,55,12,4
Organic Growth,734,262,100,14


Unknown fill (share of max), live: {'New Logo': {'web': 0.261, 'news': 0.283, 'seat': 0.25}, 'Organic Growth': {'web': 0.28, 'news': 0.32, 'seat': 0.097}}


,in both top 25,in both top 50,in both top 100
New Logo,25,46,86
Organic Growth,24,44,90


In [52]:
# INSPECT: mock signals behave like the mock: share unknown by goal (vs workbook), and score bands (vs dashboard)
sig_cols = {"web_visits_30d": "website", "news_signal": "news", "job_postings_30d": "jobs",
            "diamond_referral": "diamond", "prior_bpo_flag": "competitor", "seats_added_6m": "seat growth"}
blank_share = lambda d: d.groupby("goal")[[c for c in sig_cols if c in d]].apply(lambda x: x.isna().mean().round(2)).rename(columns=sig_cols)
display(pd.concat({"ours": blank_share(output), "mock workbook": blank_share(golden)}, axis=1))
bands = lambda s: pd.cut(s, [-1, 19, 39, 59, 79, 100], labels=["0-19", "20-39", "40-59", "60-79", "80-100"])
pd.concat({"ours": pd.crosstab(bands(output["priority_score"]), output["goal"], normalize="columns").round(2),
           "dashboard (mock)": pd.crosstab(bands(dash["priority_score"]), dash["goal"], normalize="columns").round(2)}, axis=1)

ours                                            mock workbook                               
               website  news  jobs diamond competitor seat growth       website  news  jobs diamond competitor
goal                                                                                                          
New Logo          0.68  0.55  0.68     0.6       0.58         1.0          0.69  0.55  0.70    0.59       0.61
Organic Growth    0.28  0.37  0.57     0.0       0.00         0.0          0.21  0.34  0.53    0.00       0.00

ours                dashboard (mock)               
goal           New Logo Organic Growth         New Logo Organic Growth
priority_score                                                        
0-19               0.21           0.35             0.26           0.27
20-39              0.53           0.32             0.58           0.37
40-59              0.18           0.21             0.12           0.19
60-79              0.07           0.10             0.04           0.14
80-100             0.02           0.02             0.01           0.03

## 9. Route (`route`)
Teams are the mock dashboard's three roles: **BDM**, **Marketing** and **SDR** (SDR owns New Logo only).

**1. Accounts with an Act now trigger go to the team of their first trigger** (`hot_trigger`, `TRIGGER_TEAM`), in both routing modes.
The bonus column is the trigger's bonus and family: when an account has several triggers, the best bonus of each family counts,
the 2nd family half, the 3rd a quarter (§8):

| Trigger | New Logo | Organic Growth | Bonus |
|---|---|---|---|
| Diamond referral | BDM | – | +7.5 (Referral) |
| Form fill | BDM | BDM | +15 (Website) |
| Team Builder visit | **SDR** (a prospect's visit) | BDM (a client's visit) | +15 (Website) |
| Funding round / PE buyout (30 days) | BDM | BDM | +15 (News) |
| Hiring spike or 18+ job postings | **SDR** | – | +15 (Hiring) |
| 20+ website visits in 30 days, no form fill | **Marketing** | **Marketing** | +15 (Website) |
| Restructuring / leadership-change news (30 days) | **Marketing** | **Marketing** | +15 (News) |

**2. All other accounts follow `ROUTING_MODE`** (§0b):

**`"random"` (live).** The mock dashboard's own role assignment: a **seeded random draw** with set probabilities per account type
(`ROUTING_PROBABILITIES`). The same account gets the same team on every run. It is a mock stand-in for real owner assignment, and
every row is tagged `assigned_team RANDOM (mock) unless an Act now trigger applies` in `enrichment_source`.

| Account type (no trigger) | BDM | Marketing | SDR |
|---|---|---|---|
| Organic Growth | 70% | 30% | – |
| New Logo, top 10% by score of the prospects without a trigger (`BDM_TOP_SHARE`; today a score of about 36+) | 80% | 20% | – |
| Other New Logo | – | 40% | 60% |

**`"rules"`.** Deterministic: top 10% of the goal's accounts without a trigger → BDM; Organic Growth in Tier 3 → Marketing; other New Logo → SDR; other Organic Growth → BDM.
There is **no** "no data → Marketing" rule.

Both modes use the final score (after the Act now bonus). `rank_in_team_list` = position within each team × goal list,
by `sort_key` (the score in `dashboard_v2`; the older profiles put Act now accounts first).

In [53]:
def live_triggers(S=SCORING):
    """The (trigger, goal) pairs that can fire with settings S."""
    H, out = S["hot"], set()
    if H["diamond"]:
        out.add(("Diamond referral", "New Logo"))
    if H["form_fill"]:
        out |= {("Form fill", "New Logo"), ("Form fill", "Organic Growth")}
    if H["team_builder"]:
        out.add(("Team Builder visit", "Organic Growth"))
    if H["team_builder_prospect"]:
        out.add(("Team Builder visit", "New Logo"))
    if H["top_news"]:
        out |= {(c, g) for c, v in NEWS_STRENGTH.items() if v >= H["top_news_min_strength"] and c != "None"
                for g in ("New Logo", "Organic Growth")}
    if H["hiring_spike"]:
        out.add(("Hiring spike", "New Logo"))
    if H["heavy_visits_min"] is not None:
        out |= {("Heavy website visits", "New Logo"), ("Heavy website visits", "Organic Growth")}
    out |= {(c, g) for c in H["campaign_news"] for g in ("New Logo", "Organic Growth")}
    return out

def trigger_team(r):
    """Owner team of an account's Act now trigger (BDM when the trigger isn't in TRIGGER_TEAM)."""
    return TRIGGER_TEAM.get(r["hot_trigger"], {}).get(r["goal"], "BDM")

def bdm_candidates(out, share=BDM_TOP_SHARE):
    """True for accounts WITHOUT an Act now trigger whose score is in the top `share` of their goal's accounts without a
    trigger. Ranked like the tiers: an account's position = share of those accounts scoring strictly higher, so ties share it."""
    no = out["hot_trigger"].isna()
    s, g = out.loc[no, "priority_score"], out.loc[no, "goal"]
    above = (s.groupby(g).rank(method="min", ascending=False) - 1) / g.map(g.value_counts())
    return (above < share).reindex(out.index, fill_value=False)

def bdm_cutoffs(out, share=BDM_TOP_SHARE):
    """Lowest score among each goal's BDM candidates today: {goal: score}."""
    c = bdm_candidates(out, share)
    return {g: int(out.loc[c & (out["goal"] == g), "priority_score"].min()) for g in GOALS.values() if (c & (out["goal"] == g)).any()}

def _account_type(r):
    """The account type used by ROUTING_PROBABILITIES (accounts without an Act now trigger)."""
    if r["goal"] == "Organic Growth":
        return "Organic Growth"
    return "New Logo, top 10%" if r["_bdm_candidate"] else "New Logo, other"

def route_account_random(r, seed=MOCK_SEED):
    """Mock role assignment: a seeded random draw (same team for an account on every run, whatever the row order)."""
    probs = ROUTING_PROBABILITIES[_account_type(r)]
    rng = random.Random(f"{seed}:team:{r['account_id']}")
    return rng.choices(list(probs), weights=list(probs.values()))[0]

def route_account_rules(r):
    """Deterministic routing for accounts without a trigger. There is no 'no data -> Marketing' rule."""
    nl = r["goal"] == "New Logo"
    if r["_bdm_candidate"]:                       # top BDM_TOP_SHARE of the goal's accounts without a trigger
        return "BDM"
    if not nl and r["tier"] == "Tier 3":          # weaker Organic Growth -> Marketing (nurture); BDM keeps the stronger ones
        return "Marketing"
    return "SDR" if nl else "BDM"

def route_account(r, mode=ROUTING_MODE):
    if not _blank(r["hot_trigger"]):               # an Act now trigger decides the team, in both modes
        return trigger_team(r)
    return route_account_random(r) if mode == "random" else route_account_rules(r)

def route(out, mode=ROUTING_MODE):
    """Stage 9: owner team + rank within each team x goal list."""
    out = out.copy()
    out["_bdm_candidate"] = bdm_candidates(out)
    out["assigned_team"] = out.apply(route_account, axis=1, mode=mode)
    out = out.drop(columns="_bdm_candidate")
    if mode == "random":
        out["enrichment_source"] = out["enrichment_source"] + "; assigned_team RANDOM (mock) unless an Act now trigger applies"
    out["rank_in_team_list"] = (out.groupby(["assigned_team", "goal"])["sort_key"]
                                   .rank(method="min", ascending=False).astype(int))
    return out

In [54]:
output = route(output)
pd.crosstab([output["assigned_team"], output["goal"]], output["tier"], margins=True)

tier                          Tier 1  Tier 2  Tier 3   All
assigned_team goal                                        
BDM           New Logo            14      29       5    48
              Organic Growth     108     217     457   782
Marketing     New Logo             0      21      52    73
              Organic Growth       9     113     206   328
SDR           New Logo             8      21      67    96
All                              139     401     787  1327

In [55]:
# INSPECT: score range per list, and the hot triggers
display(output.groupby(["goal", "assigned_team"])["priority_score"].describe()[["count", "min", "50%", "max"]])
output.groupby("goal")["hot_trigger"].value_counts().to_frame("accounts")

count   min   50%    max
goal           assigned_team                          
New Logo       BDM             48.0  29.0  50.5   87.0
               Marketing       73.0   4.0  23.0   48.0
               SDR             96.0   0.0  23.0   71.0
Organic Growth BDM            782.0   0.0  25.0  100.0
               Marketing      328.0   0.0  25.0   68.0

accounts
goal           hot_trigger                   
New Logo       Diamond referral            35
               Form fill                    9
               Hiring spike                 9
               Heavy website visits         7
               Team Builder visit           6
               Funding round                2
               PE buyout                    2
               Leadership change            1
Organic Growth Team Builder visit         138
               Form fill                  103
               Heavy website visits       102
               Funding round               13
               PE buyout                    9
               Restructuring                7
               Leadership change            4

In [56]:
# INSPECT: top 5 of every team list
(output.sort_values(["assigned_team", "goal", "rank_in_team_list"])
       .groupby(["assigned_team", "goal"]).head(5)
       [["assigned_team", "goal", "rank_in_team_list", "company_name", "priority_score", "tier", "hot_trigger",
         "seats_added_6m", "estimated_fields", "industry"]])

,assigned_team,goal,rank_in_team_list,company_name,priority_score,tier,hot_trigger,seats_added_6m,estimated_fields,industry
28,BDM,New Logo,1,Brodware Pty Ltd,87,Tier 1,Funding round,NaN,NaN,Construction & Engineering
75,BDM,New Logo,2,Green Oasis - Biophilic Solution,86,Tier 1,Form fill,NaN,NaN,Business & Consumer Services
208,BDM,New Logo,3,WK SERVICE,85,Tier 1,Funding round,NaN,Website,Transport & Logistics
133,BDM,New Logo,4,Ozaccom Pty Ltd,84,Tier 1,Diamond referral,NaN,News,Technology
128,BDM,New Logo,5,Nterio Inc.,78,Tier 1,Diamond referral,NaN,Website,Construction & Engineering
609,BDM,Organic Growth,1,HEMA Corporation,100,Tier 1,Team Builder visit,7.0,NaN,Construction & Engineering
355,BDM,Organic Growth,2,Brooke Institute Pty Ltd,97,Tier 1,Team Builder visit,0.0,NaN,Technology
360,BDM,Organic Growth,2,Built Holdings Pty Limited,97,Tier 1,Form fill,4.0,NaN,Technology
1087,BDM,Organic Growth,4,Scandinavian Kitchen Ltd,96,Tier 1,Team Builder visit,1.0,NaN,Financial Services
1278,BDM,Organic Growth,5,Vietell Pty Ltd,94,Tier 1,Team Builder visit,0.0,NaN,Construction & Engineering


## 10. Data quality checks: scoring and routing
Any failed `error` check stops the pipeline.

In [57]:
def run_scoring_checks(out):
    rows = []
    def check(name, level, ok, detail=""):
        rows.append({"check": name, "level": level, "passed": bool(ok), "detail": detail})
    allowed = out.apply(lambda r: r["goal"] in TEAM_GOALS.get(r["assigned_team"], []), axis=1)
    check("assigned_team is BDM / Marketing / SDR, and SDR owns New Logo only", "error",
          allowed.all(), f"{(~allowed).sum()} violations")
    check("priority_score within 0-100", "error", out["priority_score"].between(0, 100).all())
    check("tier valid", "error", out["tier"].isin(["Tier 1", "Tier 2", "Tier 3"]).all())
    hot = out["hot_trigger"].notna()
    check("Act now bonus only on hot accounts, and never lowers the score", "error",
          (out.loc[~hot, "act_now_bonus"] == 0).all() and (out["priority_score"] >= out["score_before_bonus"]).all())
    on_trigger_team = out.loc[hot].apply(lambda r: r["assigned_team"] == trigger_team(r), axis=1)
    check("every account with an Act now trigger is on that trigger's team", "error", on_trigger_team.all(),
          f"{(~on_trigger_team).sum()} violations")
    first_ok = (out.loc[hot, "act_now_triggers"].str.split("; ").str[0] == out.loc[hot, "hot_trigger"]).all() \
               and out.loc[~hot, "act_now_triggers"].isna().all()
    check("hot_trigger is the first of act_now_triggers", "error", first_ok)
    hot_share = out.groupby("goal")["hot_trigger"].apply(lambda s: s.notna().mean())
    check(f"Act now on at most {ACT_NOW_TARGET_SHARE:.0%} of each goal's accounts", "warn",
          (hot_share <= ACT_NOW_TARGET_SHARE).all(), ", ".join(f"{g} {v:.1%}" for g, v in hot_share.items()))
    if SCORING["tier"].get("method") == "percentile":
        t1 = out.groupby("goal")["tier"].apply(lambda s: (s == "Tier 1").mean())
        check("Tier 1 share per goal close to the target (ties can add a little)", "info",
              (t1 <= SCORING["tier"]["tier1_share"] + 0.05).all(), ", ".join(f"{g} {v:.1%}" for g, v in t1.items()))
    unknown_trig = sorted(set(out.loc[hot, "hot_trigger"]) - set(TRIGGER_TEAM))
    check("every trigger has an entry in TRIGGER_TEAM", "warn", not unknown_trig, f"missing: {unknown_trig}")
    og = out["goal"] == GOALS["mystaff"]
    check("Diamond / Competitor are N/A for Organic Growth", "error",
          (out.loc[og, ["diamond_referral", "prior_bpo_flag"]] == "N/A").all().all())
    check("seat growth only scored for Organic Growth", "error",
          out.loc[~og, "seat_pts"].isna().all() and out.loc[og, "seat_pts"].notna().all())
    fit_max = out["goal"].map(lambda g: SCORING["max_points"][g].get("fit", 0))
    if fit_max.any():
        check("fit points within 0 and the goal's fit maximum", "error", out["fit_pts"].between(0, fit_max).all())
        fit_known = out["fit_signal (0-1)"].notna().groupby(out["goal"]).mean()
        check("fit known (from the propensity extract)", "info", (fit_known > 0.5).all(),
              ", ".join(f"{g} {v:.0%}" for g, v in fit_known.items()))
    check("every list ranked from 1", "error",
          out.groupby(["assigned_team", "goal"])["rank_in_team_list"].min().eq(1).all())
    check("every row tagged with enrichment_source", "error", out["enrichment_source"].notna().all())
    seat_known = out.loc[og, "seats_added_6m"].notna().mean()
    check("seat growth known for Organic Growth", "info", seat_known > 0.9, f"{seat_known:.0%} of clients")
    is_mock = out["enrichment_source"].str.contains("MOCK").any()
    check("enrichment uses real data", "warn", not is_mock,
          "signals are SYNTHETIC (mock) except seat growth" + ("; team assignment is RANDOM (mock)" if ROUTING_MODE == "random" else "")
          + "; scores and routing are for testing only" if is_mock else "")
    return pd.DataFrame(rows)

In [58]:
dq_scoring = run_scoring_checks(output)
display(dq_scoring)
assert_dq(dq_scoring)
dq = pd.concat([dq, dq_scoring], ignore_index=True)

,check,level,passed,detail
0,"assigned_team is BDM / Marketing / SDR, and SDR owns New...",error,True,0 violations
1,priority_score within 0-100,error,True,
2,tier valid,error,True,
3,"Act now bonus only on hot accounts, and never lowers the...",error,True,
4,every account with an Act now trigger is on that trigger...,error,True,0 violations
5,hot_trigger is the first of act_now_triggers,error,True,
6,Act now on at most 10% of each goal's accounts,warn,False,"New Logo 32.7%, Organic Growth 33.9%"
7,Tier 1 share per goal close to the target (ties can add ...,info,True,"New Logo 10.1%, Organic Growth 10.5%"
8,every trigger has an entry in TRIGGER_TEAM,warn,True,missing: []
9,Diamond / Competitor are N/A for Organic Growth,error,True,


## 11. Dashboard data (`build_dashboard_data`)
Shapes the scored accounts into the JSON that the Account Priority Dashboard reads
(the same structure as `data/account_priority_dashboard.json`, made by the dashboard's `prep.py`):

| Part | Content |
|---|---|
| `rows` | One object per account. Pipeline columns renamed to the dashboard's names (`prior_bpo_flag` → `competitor_client`, `assigned_team` → `role`, `score_before_bonus` → `workbook_score`, …), plus display fields: `team` (role\|goal list), `chips` (the "Why" reasons), `profile`, `industry_lbl`, `band_lbl`, `estimated`, `seat_band`, `raw_score` |
| `csv_score`, `csv_tier`, `csv_team` | The same accounts scored with the **workbook** settings, for the dashboard's "what the rounded points and seat growth change" comparison |
| `settings` | `SCORING` (the live profile) flattened to the dashboard's `{key, label, value, kind}` list |
| `meta` | Title, subtitle, source, as-of date (`extract`), `generated`, the 5 lists, roles, band order, worked-example account, `enrichment_mode`, `mock_signals`, `trigger_teams` (owner team, family and bonus of each trigger), `unknown_fill` (share of max used for an unknown component, per goal), `tier_cutoffs` (lowest score in each tier, per goal), `hot_first`, `notice` (warning banner) |
| `qa` | Row counts for the dashboard's QA page (sources, merge, lists, seat data) |

While signals are mock, the subtitle and the page's warning banner (`meta.notice`) say so, and every Why chip from a mock
signal ends in **"· mock"** (seat growth chips are real, so they don't).

In [59]:
DASH_RENAME = {
    "prior_bpo_flag": "competitor_client", "bpo_pts": "competitor_pts", "assigned_team": "role",
    "score_before_bonus": "workbook_score", "web_signal (0-1)": "web_signal",
    "news_jobs_signal (0-1)": "news_jobs_signal", "seat_signal (0-1)": "seat_signal", "fit_signal (0-1)": "fit_signal",
}
DASH_ROW_FIELDS = [   # every field the dashboard page reads, plus a few keys for tracing a row back to the pipeline
    "account_id", "company_name", "goal", "industry", "employee_band", "region",
    "web_visits_30d", "team_builder_visit", "form_fill_30d", "news_signal", "news_signal_date", "job_postings_30d",
    "hiring_spike", "diamond_referral", "competitor_client", "industry_trend", "seats_added_6m",
    "industry_modifier", "web_signal", "news_jobs_signal", "seat_signal", "seat_band", "fit_signal",
    "web_pts", "news_jobs_pts", "seat_pts", "fit_pts", "diamond_pts", "competitor_pts", "base_points", "raw_score",
    "workbook_score", "workbook_tier", "priority_score", "tier", "hot_trigger", "estimated_fields", "estimated",
    "data_coverage", "sort_key", "act_now_bonus", "act_now_triggers", "csv_score", "csv_tier", "csv_team",
    "role", "team", "industry_lbl", "band_lbl", "profile", "chips",
    "shortcode", "crm_company_id", "enrichment_source",
    "cs_staff_current", "client_since", "client_years", "client_tenure_lbl",   # Organic Growth: the "With us" column
    *PROPENSITY_COLS,                          # stage 7b: the bands feed the Fit points; the page shows them as fit chips
]
SEAT_BAND_LABELS = [(0, "0"), (1, "1-5"), (6, "6-10"), (11, "11+")]

def _dots(s):
    return "●●●" if s >= CHIP_STRONG else "●●○" if s >= CHIP_MODERATE else "●○○"

SEAT_RANGE = {5: "1–4", 10: "5–9", 20: "10–19"}    # current seats for each next seat milestone
FIT_BASE_KEYS = {"fit_seat_milestone = 5": "seat_5", "fit_seat_milestone = 10": "seat_10", "fit_seat_milestone = 20": "seat_20",
                 "fit_second_dept": "second_dept", "fit_dept_expand": "dept", "fit_growth": "growth",
                 "fit_reach_10": "reach_10", "fit_branch_1y": "branch"}

def fit_base_rates(info):
    """The usual (historical) rate of each fit score, from the extract's score definitions; {} when unknown (e.g. mock)."""
    defs = (info or {}).get("definitions") or []
    return {FIT_BASE_KEYS[d["score"]]: float(d["base_rate"]) for d in defs
            if d.get("score") in FIT_BASE_KEYS and d.get("base_rate") is not None}

def _fit_chip(text, p, base, window, compare, why, facts=(), note="", usual_label="the usual rate"):
    """[text, "fit", hover] when p clears FIT["chip_min_lift"] x the usual rate (always, if the usual rate is unknown); else None.
    hover is a dict the page renders as a card: chance, usual rate, lift, comparison group, why, facts, note."""
    if _blank(p):
        return None
    lift = (float(p) / base) if base else None
    if lift is not None and FIT.get("chip_min_lift") and lift < FIT["chip_min_lift"]:
        return None
    return [text, "fit", {"chance": round(float(p), 3), "window": window, "usual": None if base is None else round(base, 3),
                          "usual_label": usual_label,
                          "lift": None if lift is None else round(lift, 1), "band": FIT["chip_band"], "compare": compare,
                          "why": None if _blank(why) else why, "facts": [[k, v] for k, v in facts if not _blank(v)],
                          "note": note or None}]

def fit_chips(a, tag="", base=None, prospect_avg=None):
    """Fit chips for one account: what it is likely to do. One chip per Client Propensity score whose band is FIT["chip_band"]
    (High) and whose chance is at least FIT["chip_min_lift"] x the usual rate. base = fit_base_rates(); prospect_avg = the
    average New Logo chance per score, used instead of base where FIT["New Logo"]["growth_compare"] = "prospects"."""
    base, hi = base or {}, FIT["chip_band"]
    out = []
    if a["goal"] == GOALS["mystaff"]:
        ms = a["fit_seat_milestone"]
        if not _blank(ms) and a["fit_seat_milestone_band"] == hi:
            ms = int(ms)
            out.append(_fit_chip(f"Likely {ms}+ seats{tag}", a["fit_seat_milestone_p"], base.get(f"seat_{ms}"),
                                 f"of reaching {ms}+ seats (held 30 days) within 12 months",
                                 f"Top third of past clients with {SEAT_RANGE.get(ms, '?')} seats", a["fit_seat_milestone_why"]))
        n = a["depts_so_far"]
        names = [] if _blank(a["depts_so_far_names"]) else a["depts_so_far_names"].split("; ")   # "; ": names can contain commas
        facts = [("Departments now", "; ".join(names[:5]) + (f" and {len(names) - 5} more" if len(names) > 5 else "")),
                 ("Often added next", a["likely_next_dept"]), ("Unusually likely", a["likely_next_dept_by_lift"])]
        if not _blank(n) and n == 1 and a["fit_second_dept_band"] == hi:
            out.append(_fit_chip(f"Likely 2nd department{tag}", a["fit_second_dept_p"], base.get("second_dept"),
                                 "of adding a second department within 12 months", "Top third of past one-department clients",
                                 a["fit_dept_why"], facts,
                                 note="Departments first staffed in a client's first 6 months count as its launch team, not as a second department."))
    else:
        note = "Scored on the prospect's industry and country only, as if we sign them (launch of 1–2 staff)."
        ten = a["fit_reach_10_band"] == hi and _fit_chip("x", a["fit_reach_10_p"], base.get("reach_10"), "", "", None)
        if ten:
            out.append(_fit_chip(f"Likely 10+ seats{tag}", a["fit_reach_10_p"], base.get("reach_10"), "of reaching 10+ seats within 2 years of signing",
                                 "Top third of past new clients", a["fit_reach_10_why"], note=note))
        elif a["fit_growth_band"] == hi:
            avg = (prospect_avg or {}).get("growth")
            profile = f"{a['industry'] or 'Industry unknown'} · {propensity_country(a['region'])}"
            past = base.get("growth")
            if avg:     # compared with the other prospects on the list, scored the same way
                out.append(_fit_chip(f"Likely 5+ seats{tag}", a["fit_growth_p"], avg, "of reaching 5+ seats within 1 year of signing",
                                     "Compared with: the other prospects on the list, scored the same way (industry and country)",
                                     a["fit_growth_why"],
                                     [("Profile", profile),
                                      ("Past new clients", None if past is None else f"{past:.0%} reached 5+ seats in their first year (their role and launch size were known)")],
                                     note=note, usual_label="the average prospect"))
            else:
                out.append(_fit_chip(f"Likely 5+ seats{tag}", a["fit_growth_p"], past, "of reaching 5+ seats within 1 year of signing",
                                     "Top third of past new clients", a["fit_growth_why"], [("Profile", profile)], note=note))
        if a["fit_branch_1y_band"] == hi:
            out.append(_fit_chip(f"Likely 2nd department{tag}", a["fit_branch_1y_p"], base.get("branch"),
                                 "of adding a second department within 1 year of signing", "Top third of past new clients",
                                 a["fit_branch_1y_why"], note=note + " Which department can't be predicted until the starting role is known."))
    return [c for c in out if c]

def dashboard_chips(a, mock, prop_mock=False, fit_base=None, prospect_avg=None):
    """The dashboard's "Why" reasons for one account: [text, kind] or [text, kind, hover], kind = signal | fit | flag | est
    (prep.py rules, plus the fit chips)."""
    tag = MOCK_CHIP_TAG if mock else ""
    out = []
    w, n, s = a["web_signal"], a["news_jobs_signal"], a["seat_signal"]
    if not _blank(w) and w > 0:
        lab = "Team Builder visit" if a["goal"] == "Organic Growth" and a["team_builder_visit"] == "Yes" else "Website"
        out.append([f"{lab} {_dots(w)}{tag}", "signal"])
    if not _blank(n) and n > 0:
        news = a["news_signal"]
        lab = news if not _blank(news) and news != "None" else "Hiring spike" if a["hiring_spike"] == "Yes" else "Job postings"
        out.append([f"{lab} {_dots(n)}{tag}", "signal"])
    if a["goal"] == "Organic Growth" and not _blank(s) and s > 0:
        out.append([f"{a['seat_band']} seats added {_dots(s)}", "signal"])          # real data (MyStaff): no mock tag
    if not _blank(a["industry_trend"]) and a["industry_trend"] >= GROWING_INDUSTRY:
        out.append([f"Growing industry{tag}", "signal"])
    if a["diamond_referral"] == "Yes":
        out.append([f"Diamond referral{tag}", "flag"])
    if a["competitor_client"] == "Yes":
        out.append([f"Competitor client{tag}", "flag"])
    if not _blank(a.get("fit_pts")):                          # fit is scored (dashboard_v3): show its chips
        out += fit_chips(a, MOCK_CHIP_TAG if prop_mock else "", fit_base, prospect_avg)
    est = a["estimated_fields"]
    if not _blank(est):
        out.append(["No " + est.lower().replace(", seat growth", " or seat growth").replace(", ", " or ") + " data", "est"])
    return out

def dashboard_settings(S=SCORING, bdm_cutoff=None):
    """The live scoring profile as the dashboard's settings list. Keys and labels match the dashboard's own JSON."""
    nl, og = S["max_points"]["New Logo"], S["max_points"]["Organic Growth"]
    T, A = S["tier"], S["act_now_bonus"]
    tier_rows = ([("tier1_share", "share", "Tier 1: top share of each goal's accounts", T["tier1_share"]),
                  ("tier2_share", "share", "Tier 2: the next share of each goal's accounts", T["tier2_share"])]
                 if T.get("method") == "percentile" else
                 [("tier1", "count", "Tier 1 minimum score", T["tier1"]), ("tier2", "count", "Tier 2 minimum score", T["tier2"])])
    stack_rows = ([("hot_bonus_stack", "share", "Trigger bonus: each further trigger family is worth this share of the one before",
                    A["stack_decay"])] if A.get("stack_decay") is not None else [])
    F = S.get("fit")
    fit_rows = ([("fit_max_nl", "points", "Fit max points, New Logo", nl["fit"]),
                 ("fit_max_og", "points", "Fit max points, Organic Growth", og["fit"]),
                 ("fit_seat_w", "share", "Fit, Organic Growth: next seat milestone share", F["Organic Growth"]["seat_weight"]),
                 ("fit_dept_w_og", "share", "Fit, Organic Growth: second department share (one-department clients)", F["Organic Growth"]["dept_weight"]),
                 ("fit_growth_w", "share", "Fit, New Logo: seat growth (5+ or 10+) share", F["New Logo"]["growth_weight"]),
                 ("fit_dept_w_nl", "share", "Fit, New Logo: second department share", F["New Logo"]["dept_weight"]),
                 ("fit_high", "share", "Fit band High: share of the fit part", F["band_value"]["High"]),
                 ("fit_medium", "share", "Fit band Medium: share of the fit part", F["band_value"]["Medium"]),
                 ("fit_low", "share", "Fit band Low: share of the fit part", F["band_value"]["Low"]),
                 ("fit_chip_lift", "factor", "Fit chip: chance at least this many times the usual rate", F.get("chip_min_lift") or 0),
                 ("base_fit", "share", "Fit baseline when unknown" + (" (fallback)" if S.get("unknown_fill") == "known_mean" else ""), S["baseline"]["fit"])]
                if F and og.get("fit") else [])
    rows = [
        ("vol_cap", "count", "Visits in 30 days for full volume credit", S["web"]["vol_cap"]),
        ("jobs_cap", "count", "Job postings for full hiring credit", S["news"]["jobs_cap"]),
        *tier_rows,
        ("bdm_min", "count", f"New Logo score for the BDM draw: top {BDM_TOP_SHARE:.0%} of prospects without a trigger event (today's lowest)",
         BDM_MIN_SCORE if bdm_cutoff is None else bdm_cutoff),
        ("bdm_top_share", "share", "New Logo prospects without a trigger event that can be drawn for BDM: the top share by score", BDM_TOP_SHARE),
        ("hot_days", "count", "Funding / PE buyout news window for a trigger event (days)", S["hot"]["top_news_days"]),
        ("hiring_min_postings", "count", "Job postings for the hiring-spike trigger event", S["hot"]["hiring_min_postings"]),
        ("heavy_visits_min", "count", "Website visits in 30 days for the heavy-visits trigger event (no form fill)", S["hot"]["heavy_visits_min"]),
        ("campaign_news_days", "count", "Restructuring / leadership-change news window for a trigger event (days)", S["hot"]["campaign_news_days"]),
        ("mod_min", "factor", "Industry multiplier at weakest trend", S["modifier"]["at_trend_0"]),
        ("mod_max", "factor", "Industry multiplier at strongest trend", S["modifier"]["at_trend_1"]),
        ("hot_strength", "factor", "Minimum news strength for a trigger event", S["hot"]["top_news_min_strength"]),
        *[(f"news_{i}", "news", cat, v) for i, (cat, v) in enumerate(NEWS_STRENGTH.items(), 1)],
        *[(f"rec_{i}", "recency", lab, f) for i, ((lo, f), lab) in
          enumerate(zip(RECENCY, ["0–30 days", "31–90 days", "91–180 days", "Over 180 days"]), 1)],
        ("hot_bonus", "points", "Trigger bonus points (added after the industry nudge, score capped at 100)", S["act_now_bonus"]["default"]),
        ("hot_bonus_diamond", "points", "Trigger bonus points when the trigger is a Diamond referral (it already adds flag points)",
         S["act_now_bonus"]["Diamond referral"]),
        *stack_rows,
        ("web_max_nl", "points", "Website max points, New Logo", nl["web"]),
        ("news_max_nl", "points", "News and jobs max points, New Logo", nl["news"]),
        ("dia_max", "points", "Diamond referral points (New Logo)", nl["diamond"]),
        ("competitor_max", "points", "Competitor client points (New Logo)", nl["bpo"]),
        ("web_max_og", "points", "Website max points, Organic Growth", og["web"]),
        ("news_max_og", "points", "News and jobs max points, Organic Growth", og["news"]),
        ("seat_max_og", "points", "Seat growth max points, Organic Growth", og["seat"]),
        ("tb_share", "share", "Website: Team Builder visit share", S["web"]["tb_share"]),
        ("ff_share", "share", "Website: form fill share", S["web"]["ff_share"]),
        ("vol_share", "share", "Website: visit volume share", S["web"]["vol_share"]),
        ("news_share", "share", "News and jobs: strongest news share", S["news"]["news_share"]),
        ("jobs_share", "share", "News and jobs: hiring share", S["news"]["jobs_share"]),
        # with unknown_fill = "known_mean" these are only the fallback; the values used are in meta.unknown_fill
        ("base_web", "share", "Website baseline when unknown" + (" (fallback)" if S.get("unknown_fill") == "known_mean" else ""), S["baseline"]["web"]),
        ("base_news", "share", "News and jobs baseline when unknown" + (" (fallback)" if S.get("unknown_fill") == "known_mean" else ""), S["baseline"]["news"]),
        ("base_seats", "share", "Seat growth baseline when unknown" + (" (fallback)" if S.get("unknown_fill") == "known_mean" else ""), S["baseline"]["seat"]),
        *[(k, "share", f"Seats added in 6 months: {lab}", share) for k, lab, (lo, share) in
          zip(["seat_0", "seat_1_5", "seat_6_10", "seat_11"], ["0", "1-5", "6-10", "11 or more"], S["seat_bands"])],
        *fit_rows,
    ]
    return [{"key": k, "kind": kind, "label": label, "value": value} for k, kind, label, value in rows]

def _json_value(v):
    """NaN / NA / NaT -> None, numpy -> Python, dates -> 'YYYY-MM-DD' (the dashboard's JSON must be strict JSON)."""
    if isinstance(v, (list, dict, str)):
        return v
    if v is None or pd.isna(v):
        return None
    if isinstance(v, pd.Timestamp):
        return v.strftime("%Y-%m-%d")
    return v.item() if hasattr(v, "item") else v

def build_dashboard_data(out, recon, as_of=SCORING_AS_OF, propensity=None):
    """Stage 11: the Account Priority Dashboard's JSON (rows, settings, meta, qa)."""
    mock = out["enrichment_source"].str.contains("MOCK").any()      # enrichment signals: the "· mock" chip tag
    prop_mock = out["propensity_source"].astype(str).str.contains("MOCK").any()   # fit chips get the tag only if propensity is mock
    any_mock = mock or prop_mock
    og = out["goal"] == GOALS["mystaff"]
    d = out.rename(columns=DASH_RENAME).copy()
    # the same accounts with the workbook settings: the dashboard's "csv" comparison columns
    wb = score_accounts(out, as_of, SCORING_WORKBOOK)
    d["csv_score"], d["csv_tier"], d["csv_team"] = wb["priority_score"], wb["tier"], wb["_workbook_team"]
    d["workbook_tier"] = tiers(d["workbook_score"], d["goal"])
    d["raw_score"] = d["base_points"] * d["industry_modifier"]
    d["seat_band"] = d["seats_added_6m"].map(
        lambda n: None if _blank(n) else next(lab for lo, lab in reversed(SEAT_BAND_LABELS) if n >= lo)).where(og)
    # Organic Growth: Diamond / Competitor are not applicable -> null flag, 0 points (as the dashboard's data has them)
    for flag, pts in [("diamond_referral", "diamond_pts"), ("competitor_client", "competitor_pts")]:
        d.loc[og, flag] = None
        d.loc[og, pts] = 0
    d["team"] = d["role"] + "|" + d["goal"]
    d["industry_lbl"] = d["industry"].fillna("Unknown")
    d["band_lbl"] = d["employee_band"].fillna("Unknown")
    d["profile"] = [" · ".join([i if isinstance(i, str) else "Industry unknown",
                                f"{b} staff" if isinstance(b, str) else "Size unknown",
                                r if isinstance(r, str) else "Region unknown"])
                    for i, b, r in zip(d["industry"], d["employee_band"], d["region"])]
    d["estimated"] = d["estimated_fields"].notna().astype(int)
    # Organic Growth "With us": current seats (MyStaff Total Staff) and time as a client (since the first staff start;
    # MyStaff's tenure band when the date is blank)
    since = pd.to_datetime(d["client_since"], errors="coerce")
    d["client_years"] = ((as_of - since).dt.days / 365.25).round(1).where(og & since.notna())
    d["client_tenure_lbl"] = [None if not o else (
        (f"{max(1, round(y * 12))} mo" if y < 1 else f"{y:.1f} yrs") if pd.notna(y) else
        (f"{b} (MyStaff)" if isinstance(b, str) and b else None))
        for o, y, b in zip(og, d["client_years"], d["client_tenure_band"])]
    d.loc[~og, ["cs_staff_current", "client_since"]] = None
    fit_base = fit_base_rates(propensity)                    # usual rate per fit score: the chips' lift threshold and hover
    nl_growth = pd.to_numeric(d.loc[d["goal"] == GOALS["hubspot"], "fit_growth_p"], errors="coerce").dropna()
    prospect_avg = ({"growth": float(nl_growth.mean())}      # New Logo 5+ seats: compared with the average prospect
                    if FIT.get("New Logo", {}).get("growth_compare") == "prospects" and len(nl_growth) else {})
    d["chips"] = [dashboard_chips(a, mock, prop_mock, fit_base, prospect_avg) for a in d.to_dict("records")]

    rows = [{k: _json_value(v) for k, v in r.items()} for r in d[DASH_ROW_FIELDS].to_dict("records")]
    rows.sort(key=lambda r: (-r["sort_key"], r["company_name"]))           # by sort_key (score; Act now first only if hot_first)

    lists = {t: int((d["team"] == t).sum()) for t in DASH_LISTS}
    seat_known = int((og & d["seats_added_6m"].notna()).sum())
    qa = [
        {"step": "HubSpot records (prospects)", "rows": int(recon.iloc[0])},
        {"step": "  merged into an existing MyStaff client", "rows": int(recon.iloc[1])},
        {"step": "  shared website domain, moved to Organic Growth", "rows": int(recon.iloc[2])},
        {"step": "MyStaff active clients", "rows": int(recon.iloc[4])},
        {"step": "All accounts", "rows": len(d)},
        {"step": "New Logo accounts", "rows": int((d["goal"] == GOALS["hubspot"]).sum())},
        {"step": "Organic Growth accounts", "rows": int(og.sum())},
        *[{"step": f"{t.split('|')[0]} · {t.split('|')[1]} list", "rows": n} for t, n in lists.items()],
        {"step": "Organic Growth accounts with seat data (MyStaff)", "rows": seat_known},
        {"step": "Organic Growth accounts with unknown seat data (average used)", "rows": int(og.sum()) - seat_known},
    ]
    meta = {
        "title": "Account Priority Dashboard",
        "subtitle": (("Real accounts · signals are MOCK except seat growth" + ("" if prop_mock or not d["fit_pts"].notna().any() else " and fit"))
                     if mock else "Real accounts"),
        "source": f"format_priority_data pipeline v{PIPELINE_VERSION}: " + " + ".join(v["path"].name for v in SOURCES.values()),
        "extract": as_of.strftime("%Y-%m-%d"),
        "generated": pd.Timestamp.now().isoformat(timespec="seconds"),
        "shown_teams": DASH_LISTS,
        "roles": DASH_ROLES,
        "band_order": REF_BANDS,
        "default_example": rows[0]["account_id"],
        "enrichment_mode": ENRICHMENT_MODE,
        "mock_signals": bool(any_mock),
        "trigger_teams": [   # which team owns each live trigger event, its family and bonus (shown on the page's Trigger event card)
            {"trigger": t, "goal": g, "team": TRIGGER_TEAM.get(t, {}).get(g, "BDM"), "family": TRIGGER_FAMILY.get(t, t),
             "bonus": SCORING["act_now_bonus"].get(t, SCORING["act_now_bonus"]["default"])}
            for t, g in sorted(live_triggers(SCORING), key=lambda x: (list(TRIGGER_TEAM).index(x[0]) if x[0] in TRIGGER_TEAM else 99, x[1]))],
        "scoring_profile": SCORING_PROFILE,
        "hot_first": bool(SCORING.get("hot_first", True)),
        # share of max used for an unknown component, per goal (the flat baseline when the profile doesn't use known_mean)
        "unknown_fill": (known_mean_fill(out.to_dict("records"), as_of, SCORING) if SCORING.get("unknown_fill") == "known_mean"
                         else {g: dict(SCORING["baseline"]) for g in GOALS.values()}),
        # lowest final score in each tier, per goal (with percentile tiers the cut-off differs by goal)
        "tier_cutoffs": {g: {t: int(s.min()) for t, s in d.loc[d["goal"] == g].groupby("tier")["priority_score"]}
                         for g in GOALS.values()},
        "notice": (   # the dashboard's warning banner (HTML)
            "Real accounts from HubSpot (New Logo) and MyStaff (Organic Growth). <b>Website, news, jobs, Diamond referral, "
            "competitor client and industry trend signals are MOCK values made up for testing, so scores, tiers and lists are "
            "not real yet.</b> Why chips marked \"· mock\" come from them. Seat growth is real (MyStaff). "
            + ("Fit is real (Client Propensity); hover a fit chip for its reason. " if not prop_mock and d["fit_pts"].notna().any() else "")
            + ("Team (role) assignment is RANDOM, not a real owner, except accounts with a trigger event, which go to the trigger's team. " if ROUTING_MODE == "random" else "")
            + "Do not use these lists to contact accounts." if mock else
            "Real accounts from HubSpot (New Logo) and MyStaff (Organic Growth), scored by the format_priority_data pipeline.")
            + propensity_notice(propensity),
        "propensity": propensity_meta(propensity),   # stage 7b: mode, source, as-of date, model AUCs, band cut-offs
    }
    return {"meta": meta, "settings": dashboard_settings(bdm_cutoff=bdm_cutoffs(out).get(GOALS["hubspot"])), "qa": qa, "rows": rows}

def run_dashboard_checks(data, out):
    rows_ = []
    def check(name, level, ok, detail=""):
        rows_.append({"check": name, "level": level, "passed": bool(ok), "detail": detail})
    rows = data["rows"]
    check("one dashboard row per account", "error", len(rows) == len(out) and len({r["account_id"] for r in rows}) == len(rows))
    check("every row on exactly one shown list", "error", all(r["team"] in data["meta"]["shown_teams"] for r in rows))
    missing = sorted({f for r in rows for f in DASH_ROW_FIELDS if f not in r})
    check("every row has every field the page reads", "error", not missing, f"missing: {missing}")
    keys = {s["key"] for s in data["settings"]}
    needed = {"web_max_nl", "web_max_og", "news_max_nl", "news_max_og", "dia_max", "competitor_max", "seat_max_og",
              "tb_share", "ff_share", "vol_share", "vol_cap", "news_share", "jobs_share", "jobs_cap", "base_web",
              "base_news", "base_seats", "mod_min", "mod_max", "bdm_min", "hot_days",
              "hot_bonus", "hot_bonus_diamond", "seat_0", "seat_1_5", "seat_6_10", "seat_11",
              "hiring_min_postings", "heavy_visits_min", "campaign_news_days"} \
             | {f"news_{i}" for i in range(1, 9)} | {f"rec_{i}" for i in range(1, 5)}
    tier_keys = {"tier1_share", "tier2_share"} if SCORING["tier"].get("method") == "percentile" else {"tier1", "tier2"}
    needed |= tier_keys
    if SCORING.get("fit"):
        needed |= {"fit_max_nl", "fit_max_og", "fit_seat_w", "fit_dept_w_og", "fit_growth_w", "fit_dept_w_nl", "fit_high",
                   "fit_medium", "fit_low", "base_fit"}
        chips_ok = all(len(c) in (2, 3) and (c[1] != "fit" or (len(c) == 3 and isinstance(c[2], dict) and c[2].get("chance") is not None))
                       for r in rows for c in r["chips"])
        check("every fit chip carries its hover text", "error", chips_ok)
    check("every setting the page reads is present", "error", needed <= keys, f"missing: {sorted(needed - keys)}")
    meta_needed = {"title", "subtitle", "source", "extract", "generated", "shown_teams", "roles", "band_order",
                   "default_example", "notice", "trigger_teams", "unknown_fill", "tier_cutoffs", "hot_first"}
    check("meta has every key the page reads", "error", meta_needed <= set(data["meta"]),
          f"missing: {sorted(meta_needed - set(data['meta']))}")
    check("worked-example account exists", "error", any(r["account_id"] == data["meta"]["default_example"] for r in rows))
    try:
        json.dumps(data, allow_nan=False, ensure_ascii=False)
        ok = True
    except (ValueError, TypeError):
        ok = False
    check("data is strict JSON (no NaN / dates / numpy types)", "error", ok)
    check("list sizes add up to all accounts", "error",
          sum(q["rows"] for q in data["qa"] if q["step"].endswith(" list")) == len(rows))
    return pd.DataFrame(rows_)

In [60]:
dashboard_data = build_dashboard_data(output, recon, propensity=propensity_info)
dq_dashboard = run_dashboard_checks(dashboard_data, output)
display(dq_dashboard)
assert_dq(dq_dashboard)
dq = pd.concat([dq, dq_dashboard], ignore_index=True)
print(dashboard_data["meta"]["subtitle"], "|", len(dashboard_data["rows"]), "rows |", dashboard_data["meta"]["shown_teams"])
pd.DataFrame(dashboard_data["qa"])

,check,level,passed,detail
0,one dashboard row per account,error,True,
1,every row on exactly one shown list,error,True,
2,every row has every field the page reads,error,True,missing: []
3,every setting the page reads is present,error,True,missing: []
4,meta has every key the page reads,error,True,missing: []
5,worked-example account exists,error,True,
6,data is strict JSON (no NaN / dates / numpy types),error,True,
7,list sizes add up to all accounts,error,True,


Real accounts · signals are MOCK except seat growth | 1327 rows | ['Marketing|New Logo', 'Marketing|Organic Growth', 'SDR|New Logo', 'BDM|New Logo', 'BDM|Organic Growth']


,step,rows
0,HubSpot records (prospects),484
1,merged into an existing MyStaff client,265
2,"shared website domain, moved to Organic Growth",2
3,MyStaff active clients,1108
4,All accounts,1327
5,New Logo accounts,217
6,Organic Growth accounts,1110
7,Marketing · New Logo list,73
8,Marketing · Organic Growth list,328
9,SDR · New Logo list,96


In [61]:
# INSPECT: a few dashboard rows as the page will see them (Why chips, profile, list)
pd.DataFrame(dashboard_data["rows"])[["account_id", "company_name", "team", "priority_score", "tier", "workbook_score",
                                      "csv_score", "hot_trigger", "profile", "chips"]].head(12)

,account_id,company_name,team,priority_score,tier,workbook_score,csv_score,hot_trigger,profile,chips
0,MS-HEMACORP,HEMA Corporation,BDM|Organic Growth,100,Tier 1,82,87,Team Builder visit,Construction & Engineering · Size unknown · US,"[[Team Builder visit ●●● · mock, signal], [Funding round..."
1,MS-BROOKE,Brooke Institute Pty Ltd,BDM|Organic Growth,97,Tier 1,74,90,Team Builder visit,Technology · 11-50 staff · AU-SA,"[[Team Builder visit ●●● · mock, signal], [PE buyout ●●●..."
2,MS-BUILT,Built Holdings Pty Limited,BDM|Organic Growth,97,Tier 1,74,85,Form fill,Technology · Size unknown · AU-NSW,"[[Team Builder visit ●●● · mock, signal], [Funding round..."
3,MS-SCANDIKITCHEN,Scandinavian Kitchen Ltd,BDM|Organic Growth,96,Tier 1,73,82,Team Builder visit,Financial Services · 11-50 staff · UK,"[[Team Builder visit ●●● · mock, signal], [PE buyout ●●●..."
4,MS-VIETELL,Vietell Pty Ltd,BDM|Organic Growth,94,Tier 1,71,87,Team Builder visit,Construction & Engineering · Size unknown · AU-NSW,"[[Team Builder visit ●●● · mock, signal], [PE buyout ●●●..."
5,MS-HIGHLANDFIN,Highland Financial Services Pty Ltd,BDM|Organic Growth,91,Tier 1,69,83,Team Builder visit,Technology · 1-10 staff · AU-NSW,"[[Team Builder visit ●●● · mock, signal], [PE buyout ●●●..."
6,MS-MAXLENDFINANCE,MAXLEND FINANCE PTY LTD (Archived),BDM|Organic Growth,91,Tier 1,69,83,Team Builder visit,Manufacturing & Industrial · Size unknown · AU-QLD,"[[Team Builder visit ●●● · mock, signal], [PE buyout ●●●..."
7,MS-NEWFARMRE,Place New Farm,BDM|Organic Growth,90,Tier 1,68,83,Form fill,Accounting · Size unknown · AU-QLD,"[[Team Builder visit ●●● · mock, signal], [Funding round..."
8,MS-REZZIPFND,PFND PTY LTD,BDM|Organic Growth,89,Tier 1,74,92,Form fill,Real Estate · Size unknown · AU-WA,"[[Team Builder visit ●●● · mock, signal], [M&A ●●● · moc..."
9,MS-WIPFLI,Wipfli Advisory LLC,BDM|Organic Growth,89,Tier 1,74,86,Form fill,"Marketing, Media & Creative · Size unknown · US","[[Team Builder visit ●●● · mock, signal], [Expansion / n..."


## 12. Load (`load_output`, `load_dashboard`)
Saves the **processed data** to the local data folder, **not** the repo, because it contains confidential client data:
* `priority_company_info_combined.xlsx`: every column, colour-coded by group, plus matches, rejects, DQ checks, settings and run info
* `account_priority_dashboard_data.json`: the dashboard's data (stage 11), read by `../build_dashboard/build_dashboard.py`

While signals are mock, both file names get a `_MOCK_SIGNALS` suffix.

In [62]:
GROUP_FILL = {"1_company": "BDD7EE", "2_enrichment": "FFF2CC", "3_scoring": "E7E6E6", "4_lineage": "E2EFDA",
              "5_propensity": "FCE4D6"}

def load_output(out, matches, rejects, dq, path=OUTPUT_PATH, propensity=None):
    """Stage 12a: write the output workbook."""
    from openpyxl.styles import PatternFill, Font
    if is_mock_run():                        # a file with synthetic signals must never look like a real one
        path = path.with_name(path.stem + "_MOCK_SIGNALS" + path.suffix)
    groups = pd.DataFrame({"column": OUTPUT_COLS, "group": [COLUMN_GROUPS[c] for c in OUTPUT_COLS]})
    groups["description"] = groups["group"].map(GROUP_DESCRIPTION)
    groups["dashboard_extension"] = groups["column"].isin(DASHBOARD_EXTENSIONS)
    meta = pd.Series({"pipeline_version": PIPELINE_VERSION, "run_date": RUN_DATE.date(),
                      "enrichment_mode": ENRICHMENT_MODE, "scoring_profile": SCORING_PROFILE,
                      "scoring_as_of": SCORING_AS_OF.date(),
                      "propensity_mode": PROPENSITY_MODE,
                      "propensity_source": (propensity or {}).get("source", ""),
                      "propensity_file": (propensity or {}).get("accounts_file", ""),
                      **{f"source_{k}": v["path"].name for k, v in SOURCES.items()}}).to_frame("value")
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        out.to_excel(xw, sheet_name="priority_list", index=False)
        groups.to_excel(xw, sheet_name="column_groups", index=False)
        matches.to_excel(xw, sheet_name="hubspot_mystaff_matches", index=False)
        rejects.to_excel(xw, sheet_name="mystaff_rejects", index=False)
        dq.to_excel(xw, sheet_name="dq_checks", index=False)
        pd.json_normalize(SCORING, sep=".").T.rename(columns={0: "value"}).astype(str).to_excel(xw, sheet_name="scoring_settings")
        meta.to_excel(xw, sheet_name="run_info")
        if (propensity or {}).get("definitions"):    # what each propensity (fit) score means
            pd.DataFrame(propensity["definitions"]).to_excel(xw, sheet_name="propensity_definitions", index=False)
        ws = xw.sheets["priority_list"]
        for i, col in enumerate(out.columns, start=1):
            ws.cell(row=1, column=i).fill = PatternFill("solid", fgColor=GROUP_FILL[COLUMN_GROUPS[col]])
            ws.cell(row=1, column=i).font = Font(bold=True)
        ws.freeze_panes = "C2"
    print("saved ->", path)
    return path

def _mock_suffix(path):
    return path.with_name(path.stem + "_MOCK_SIGNALS" + path.suffix) if is_mock_run() else path

def load_dashboard(data, json_path=DASHBOARD_JSON_PATH):
    """Stage 12b: save the dashboard JSON locally. Build the page with ../build_dashboard/build_dashboard.py."""
    json_path = _mock_suffix(json_path)
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, allow_nan=False)
    print("saved ->", json_path)
    print("next: python build_dashboard/build_dashboard.py   (from the 'Marketing Intelligence' folder)")
    return json_path

if SAVE_OUTPUT:
    load_output(output, matches, ms_rejects, dq, propensity=propensity_info)
    load_dashboard(dashboard_data)

## Run end to end (`run_pipeline`)
Stages 1–12 in one call, which is how a scheduled job would run it. Off by default so it doesn't repeat the slow MyStaff extract.

In [63]:
def run_pipeline(save=False):
    # 1-2  extract + standardize
    stg_hs = standardize_hubspot(extract_hubspot())
    stg_ms, rejects = standardize_mystaff(extract_mystaff())
    # 3-5  match, merge, conform
    matches = match_sources(stg_hs, stg_ms)
    merged, recon = merge_sources(stg_hs, stg_ms, matches)
    out = conform_to_output(merged)
    # 6    DQ: merge
    dq = run_dq_checks(out, stg_hs, stg_ms)
    assert_dq(dq)
    # 7    signals
    out = add_signals(out)
    # 7b   propensity (Client Propensity extract, or mock / off)
    out, propensity_info = add_propensity(out)
    dq = pd.concat([dq, run_propensity_checks(out, propensity_info)], ignore_index=True)
    assert_dq(dq)
    # 8-9  score, route
    out = route(score(out))
    # 10   DQ: scoring and routing
    dq = pd.concat([dq, run_scoring_checks(out)], ignore_index=True)
    assert_dq(dq)
    # 11   dashboard data
    dash_data = build_dashboard_data(out, recon, propensity=propensity_info)
    dq = pd.concat([dq, run_dashboard_checks(dash_data, out)], ignore_index=True)
    assert_dq(dq)
    # 12   load
    if save:
        load_output(out, matches, rejects, dq, propensity=propensity_info)
        load_dashboard(dash_data)
    return out, dq, recon, dash_data

RUN_END_TO_END = True
if RUN_END_TO_END:
    output, dq, recon, dashboard_data = run_pipeline(save=SAVE_OUTPUT)

mystaff: dropped 1,045,621 empty spreadsheet rows
